# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
%cd Hierarchical-Localization
!pip install --progress-bar off --quiet -e .
!pip install --progress-bar off --quiet --upgrade plotly
# only build with ONNX gpu + caspar + ceres cuda/cudss; downloads disabled, so model paths must be set
!pip uninstall -y --quiet pycolmap
!pip install --progress-bar off --quiet \
   "https://github.com/lyehe/build_gpu_colmap/releases/download/v4.1.0/pycolmap-4.1.0+cu128.bundled.cudss-cp312-cp312-manylinux_2_35_x86_64.whl"

In [ ]:
from tqdm import tqdm
from pathlib import Path
import hashlib
import json
import numpy as np
import os
import shutil
import subprocess
import urllib.request


from hloc import reconstruction, visualization
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d
import pycolmap

In [ ]:
DATA_PATH = next(p for p in [Path(os.environ.get('SRS_HLOC_DATA') or '/nonexistent'),
                             Path('/kaggle/input/datasets/yu5uf5/buggy-hloc'),
                             Path('/kaggle/input/buggy-hloc')] if p.exists())
outputs = Path('/kaggle/working/multi') if Path('/kaggle').exists() \
    else DATA_PATH / 'outputs'
outputs.mkdir(parents=True, exist_ok=True)
IMAGES_PATH = Path('/tmp/images')

In [ ]:
# camera position relative to the racebox in body frame, meters: {roll_id: (forward, left)}
# +forward = camera ahead of the racebox, +left = camera to its left
RB_CAM_OFFSET = {
    37: (1.25, 0), 38: (1.25, 0),
    39: (1.20, 0),
    44: (-0.80, 0), 45: (-0.80, 0),
    1387: (0.05, 0),
    1388: (-0.75, 0),
    1401: (-0.70, 0)
}

R_EARTH = 6371000.0

def apply_rb_cam_offset(gps, fwd, left):
    """Move racebox positions to the camera using gps-derived heading."""
    lat = np.array([s['lat'] for s in gps], float)
    lon = np.array([s['long'] for s in gps], float)
    latr = np.radians(lat)
    i = np.arange(len(gps))
    i0, i1 = np.maximum(i - 12, 0), np.minimum(i + 12, len(gps) - 1)  # ~0.5 s window at 25 Hz
    dn = np.radians(lat[i1] - lat[i0]) * R_EARTH
    de = np.radians(lon[i1] - lon[i0]) * R_EARTH * np.cos(latr)
    ok = np.hypot(de, dn) > 0.5
    if not ok.any():
        return
    head = np.arctan2(de, dn)
    last = np.maximum.accumulate(np.where(ok, i, -1))
    last[last < 0] = np.flatnonzero(ok)[0]  # hold heading through standstill
    head = head[last]
    off_e = fwd * np.sin(head) - left * np.cos(head)
    off_n = fwd * np.cos(head) + left * np.sin(head)
    for s, oe, on, phi in zip(gps, off_e, off_n, latr):
        s['lat'] += np.degrees(on / R_EARTH)
        s['long'] += np.degrees(oe / (R_EARTH * np.cos(phi)))

In [ ]:
import cv2
from collections import defaultdict
from itertools import combinations
from scipy.spatial import cKDTree


def load_vid_imu(vid_imu_path):
    """Load vid_imu exports; shift racebox onto the (offset-corrected) frame timeline
    and move it to the camera."""
    data = {}
    for p in sorted(vid_imu_path.glob('*.json')):
        with open(p) as f:
            data[p.stem] = json.load(f)
    for run, d in data.items():
        # racebox offset is imu-estimated in smooth.ipynb's export; TIME_OFFSETS holds the
        # measured residual frame-time error (shift gps by -dt == interp at frame_ts + dt)
        off_ns = (d.get('racebox_offset_ms', 0.0) - TIME_OFFSETS.get(run, 0.0)) * 1e6
        for key in ('racebox_gps', 'racebox_speed'):
            for s in d.get(key, []):
                s['timestamp'] += off_ns
        fwd, left = RB_CAM_OFFSET.get(int(run), (0.0, 0.0))
        if fwd or left:
            apply_rb_cam_offset(d.get('racebox_gps', []), fwd, left)
    return data


def sharpness_score(bgr):
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (426, 240))
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


def extract_frames(video, start_ns, out_dir, stride=1, window_ns=None):
    """Save frames as <ts_ns>.jpg plus sharpness.json; skips folders already done."""
    if (out_dir / 'sharpness.json').exists():
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")
    scores = {}
    i = 0
    try:
        with tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc=out_dir.name, leave=False) as progress:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                progress.update(1)
                if i % stride != 0:
                    continue
                ts_ns = start_ns + int(round(cap.get(cv2.CAP_PROP_POS_MSEC) * 1_000_000))
                if window_ns and not window_ns[0] <= ts_ns <= window_ns[1]:
                    continue
                cv2.imwrite(str(out_dir / f"{ts_ns}.jpg"), frame)
                scores[ts_ns] = sharpness_score(frame)
    finally:
        cap.release()
    with open(out_dir / 'sharpness.json', 'w') as f:
        json.dump(scores, f)
    print(f"{out_dir.name}: saved {len(scores)} frames")


def gps_enu(d, field):
    gps = d[field]
    ts, lat, lon, alt = (np.array([s[k] for s in gps], float)
                         for k in ('timestamp', 'lat', 'long', 'alt'))
    enu = np.array(gt.ellipsoid_to_enu(list(np.stack([lat, lon, alt], 1)), LAT0, LON0, ALT0))
    enu[:, 2] += CAM_HEIGHT
    return ts, enu


def speed_arrays(d, kind):
    if kind == 'racebox':
        spd = d['racebox_speed']
        return (np.array([s['timestamp'] for s in spd], float),
                np.array([s['speed'] for s in spd], float))
    vel = d['velocity']
    return (np.array([s['timestamp'] for s in vel], float),
            np.hypot([s['vx'] for s in vel], [s['vy'] for s in vel]))


def select_frames(run, d, gps_kind, delta_s):
    """Distance-uniform selection over the video ∩ gps window.
    Returns ({names, enu, heading}, (gps_ts, enu_src))."""
    image_paths = sorted((IMAGES_PATH / run).glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    gps_ts, enu_src = gps_enu(d, 'racebox_gps' if gps_kind == 'racebox' else 'gps_data')
    spd_ts, spd = speed_arrays(d, gps_kind)

    m = (spd_ts >= max(gps_ts[0], image_ts[0])) & (spd_ts <= min(gps_ts[-1], image_ts[-1]))
    ts_w, v_w = spd_ts[m], spd[m]
    dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
    sel_ts = np.interp(np.arange(0.0, dist[-1], delta_s), dist, ts_w)

    # sharpest frame within each target's window (blur is vibration-driven, varies frame to frame)
    scores = {}
    score_file = IMAGES_PATH / run / 'sharpness.json'
    if score_file.exists():
        with open(score_file) as f:
            scores = {int(k): v for k, v in json.load(f).items()}
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2],
                             (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
        elif scores:
            idx.append(i0 + int(np.argmax([scores.get(int(u), 0.0) for u in image_ts[i0:i1]])))
        else:
            idx.append(i0 + np.abs(image_ts[i0:i1] - t).argmin())
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, gps_ts, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    print(f'{run}: {len(idx)} frames over {dist[-1]:.0f}m')
    return ({'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
             'enu': enu,
             'heading': np.arctan2(grad[:, 1], grad[:, 0])},
            (gps_ts, enu_src))


def sequential_pairs(names, k):
    return {(names[i], names[j])
            for i in range(len(names))
            for j in range(i + 1, min(i + 1 + k, len(names)))}


def skip_pairs(names, offsets):
    """Intra-run pairs at fixed frame offsets. At DELTA_S spacing offset o is a
    ~o*DELTA_S m baseline, so a spread of offsets spreads triangulation angles."""
    return {tuple(sorted((names[i], names[i + o])))
            for o in offsets for i in range(len(names) - o)}


def map_selection(model):
    """{run: names/enu/heading} for a solved model, for gating against its images."""
    if not isinstance(model, pycolmap.Reconstruction):
        model = pycolmap.Reconstruction(str(model))
    by_run = defaultdict(list)
    for i in model.reg_image_ids():
        im = model.images[i]
        by_run[Path(im.name).parts[0]].append((int(Path(im.name).stem), im.name,
                                               im.projection_center()))
    out = {}
    for run, items in by_run.items():
        items.sort()
        enu = np.array([c for _, _, c in items])
        grad = np.gradient(enu[:, :2], axis=0)
        out[run] = {'names': [n for _, n, _ in items], 'enu': enu,
                    'heading': np.arctan2(grad[:, 1], grad[:, 0])}
    return out


def cross_pairs(sel_a, sel_b, k, r, heading_max_deg):
    """Proximity pairs between two selections, gated on heading difference."""
    tree = cKDTree(sel_b['enu'][:, :2])
    dists, nbrs = tree.query(sel_a['enu'][:, :2], k=k, distance_upper_bound=r)
    if k == 1:
        dists, nbrs = dists[:, None], nbrs[:, None]
    pairs = set()
    for i, (ds, js) in enumerate(zip(dists, nbrs)):
        for dist, j in zip(ds, js):
            if not np.isfinite(dist):
                continue
            dh = abs(sel_a['heading'][i] - sel_b['heading'][j])
            if min(dh, 2 * np.pi - dh) <= np.radians(heading_max_deg):
                pairs.add(tuple(sorted((sel_a['names'][i], sel_b['names'][j]))))
    return pairs

In [ ]:
MODELS_PATH = Path('/tmp/models')
MASKS_PATH = Path('/tmp/masks')
LG_FP16 = True     # fp16 matcher: 1.8x pairs/s on a T4, poses agree to 4 cm p90
LG_FP16_CACHE = os.environ.get('SRS_LG_FP16_CACHE')   # optional dir or rclone target to stage from


def lightglue_model():
    """Matcher weights: fp16, converted from the fp32 asset in seconds; fp32 if LG_FP16 off."""
    f32 = MODELS_PATH / 'aliked-lightglue.onnx'
    if not LG_FP16:
        return f32
    f16 = MODELS_PATH / 'aliked-lightglue-fp16.onnx'
    if f16.exists():
        return f16
    if LG_FP16_CACHE:
        src = f'{LG_FP16_CACHE.rstrip("/")}/{f16.name}'
        if Path(src).exists():
            shutil.copy(src, f16)
        elif shutil.which('rclone'):
            subprocess.run(['rclone', 'copyto', src, str(f16)])
    if not f16.exists():
        import onnx
        # onnxconverter_common.float16 1.16.0 crashes on this graph; onnxruntime's works
        from onnxruntime.transformers.float16 import convert_float_to_float16
        onnx.save(convert_float_to_float16(onnx.load(f32), keep_io_types=True,
                                           disable_shape_infer=True), f16)
    # the measured fp16 accuracy is one conversion's; make a library change visible
    print(f'{f16.name}: sha256 {hashlib.sha256(f16.read_bytes()).hexdigest()}')
    return f16


def stage_models():
    # this pycolmap build can't auto-download onnx models; stage them locally
    MODELS_PATH.mkdir(exist_ok=True)
    for name in ('aliked-n16rot.onnx', 'aliked-lightglue.onnx'):
        f = MODELS_PATH / name
        if not f.exists():
            urllib.request.urlretrieve(
                f'https://github.com/colmap/colmap/releases/download/3.13.0/{name}', f)
    lightglue_model()


def link_masks(names, mask_src=None):
    # colmap masks: <mask_path>/<image name>.png, black = exclude; mask0 unless given one
    MASKS_PATH.mkdir(exist_ok=True)
    mask_src = Path(mask_src) if mask_src else MASKS_PATH / 'mask0.png'
    if not mask_src.exists():
        shutil.copy(DATA_PATH / 'mask0.png', mask_src)
    for ref in names:
        dst = MASKS_PATH / f'{ref}.png'
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.link(mask_src, dst)


def extract_image_features(db_path, names, camera_params=None, masks=True,
                           max_num_features=4096):
    options = pycolmap.FeatureExtractionOptions()
    options.type = pycolmap.FeatureExtractorType.ALIKED_N16ROT
    options.aliked.n16rot_model_path = str(MODELS_PATH / 'aliked-n16rot.onnx')
    options.aliked.max_num_features = max_num_features
    options.use_gpu = True
    options.gpu_index = GPU_INDEX
    reader = {'camera_model': 'SIMPLE_RADIAL',
              'camera_params': ','.join(str(v) for v in (camera_params or CAMERA_PARAMS))}
    if masks:
        reader['mask_path'] = str(MASKS_PATH)
    pycolmap.extract_features(
        db_path, IMAGES_PATH, image_names=sorted(names),
        camera_mode=pycolmap.CameraMode.PER_FOLDER,  # consider PER_IMAGE because of stabilization
        reader_options=reader,
        extraction_options=options)


def match_pairs(db_path, pairs_file):
    matching_options = pycolmap.FeatureMatchingOptions()
    matching_options.type = pycolmap.FeatureMatcherType.ALIKED_LIGHTGLUE
    matching_options.aliked.lightglue.model_path = str(lightglue_model())
    matching_options.use_gpu = True
    matching_options.gpu_index = GPU_INDEX
    pairing_options = pycolmap.ImportedPairingOptions()
    pairing_options.match_list_path = str(pairs_file)
    pycolmap.match_image_pairs(db_path, matching_options=matching_options,
                               pairing_options=pairing_options)


def write_pose_priors(db_path, gps_by_run, cov):
    """Position priors for images of runs in gps_by_run; inert unless use_prior_position."""
    with pycolmap.Database.open(str(db_path)) as db:
        have = {p.corr_data_id.id for p in db.read_all_pose_priors()}
        for image in db.read_all_images():
            p = Path(image.name)
            if image.data_id.id in have or p.parts[0] not in gps_by_run:
                continue
            gps_ts, enu = gps_by_run[p.parts[0]]
            ts = int(p.stem)
            prior = pycolmap.PosePrior()
            prior.corr_data_id = image.data_id
            prior.position = np.array([np.interp(ts, gps_ts, enu[:, i]) for i in range(3)])
            prior.position_covariance = cov
            prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
            db.write_pose_prior(prior)
        print('pose priors:', db.num_pose_priors())

In [ ]:
videos = {v.stem: v for v in DATA_PATH.glob('*.mp4')}
data = load_vid_imu(DATA_PATH / 'vid_imu')

# Create Images

In [ ]:
for stem, video in tqdm(videos.items(), desc='videos'):
    extract_frames(video, data[stem]['camera_start'], IMAGES_PATH / stem)

# Config

In [ ]:
sfm_pairs = outputs / 'pairs-sfm.txt'
sfm_dir = outputs / 'sfm'
sfm_prior_dir = outputs / 'sfm_prior'
database = outputs / 'database.db'

RESUME = False     # continue from outputs/database.db + outputs/prior
RECON_RUNS = None  # runs to reconstruct this pass; None = all

In [ ]:
# racebox: 25Hz + scalar doppler speed; fit: 10Hz + velocity vectors
GPS_KIND = 'racebox'
LAT0, LON0, ALT0 = 40.44163016, -79.94165829, 288.42151354  # shared ENU reference
# pycolmap fixed the GPSTransfrom typo after 4.1.0; the cu128 wheel still has it
_ellipsoid = getattr(pycolmap, 'GPSTransformEllipsoid', None) \
    or pycolmap.GPSTransfromEllipsoid
gt = pycolmap.GPSTransform(_ellipsoid.WGS84)

DELTA_S = 2.0         # m between selected frames
SEQ_K = 6             # forward sequential pairs per frame
CROSS_K = 3           # nearest cross-run candidates per frame
CROSS_R = 6.0         # m, max cross-run pair distance
HEADING_MAX_DEG = 40  # max cross-run heading difference

CAM_HEIGHT = 0.35  # m, gps alt is DEM road level; lift priors approximatley to the camera

# measured frame-time offsets, ms (dt grid-fit of model/localized poses vs racebox):
# gps interp at frame_ts + dt fits best; load_vid_imu applies it by shifting the gps streams
TIME_OFFSETS = {'37': -60.0, '38': -70.0, '45': -60.0, '1388': 60.0, '1401': 0.0,
                '39': -50.0, '44': -50.0, '1387': -1300.0}

In [ ]:
import torch

GPU_INDEX = ','.join(str(i) for i in range(torch.cuda.device_count()))

# caspar BA only supports SIMPLE_RADIAL [f, cx, cy, k1]/PINHOLE
# initial estimates refined per image
CAMERA_PARAMS = [653.4, 631.72, 338.74, -0.0526]

# Mapping

## Select Frames

In [ ]:
runs = sorted(data, key=int)
sel, run_enu = {}, {}
for run in runs:
    sel[run], run_enu[run] = select_frames(run, data[run], GPS_KIND, DELTA_S)

# selection must reproduce db names (guards param drift across sessions)
if RESUME:
    with pycolmap.Database.open(str(database)) as db:
        db_names = {im.name for im in db.read_all_images()}
    for run in runs:
        mine = {n for n in db_names if n.startswith(f'{run}/')}
        assert not mine or mine == set(sel[run]['names']), run

references = [n for run in runs for n in sel[run]['names']]
recon_names = sorted(n for r in (RECON_RUNS or runs) for n in sel[r]['names'])
len(references)

In [ ]:
import matplotlib.pyplot as plt

for run in runs:
    plt.plot(*sel[run]['enu'][:, :2].T, '.', ms=2, label=run)
plt.axis('equal')
plt.legend(markerscale=5)
plt.show()

In [ ]:
sl = slice(200, 210)
plot_images([read_image(IMAGES_PATH / ref) for ref in references[sl]],
            titles=references[sl], dpi=25)

## Features

In [ ]:
stage_models()
link_masks(references)

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, outputs / 'colmap.LOG.')
if not RESUME:
    database.unlink(missing_ok=True)
extract_image_features(database, references)

## Matching

In [ ]:
pairs = set()
for run in runs:
    pairs |= sequential_pairs(sel[run]['names'], SEQ_K)
n_seq = len(pairs)

for a, b in combinations(runs, 2):
    pairs |= cross_pairs(sel[a], sel[b], CROSS_K, CROSS_R, HEADING_MAX_DEG)

sfm_pairs.write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(f'{n_seq} sequential + {len(pairs) - n_seq} cross-run pairs')

In [ ]:
match_pairs(database, sfm_pairs)

## Reconstruct

### Database

In [ ]:
PRIOR_STD_XY = 0.5
PRIOR_STD_Z = 1.0
# inert unless use_prior_position is set, so both reconstructions can share the db
write_pose_priors(database, run_enu,
                  np.diag([PRIOR_STD_XY**2, PRIOR_STD_XY**2, PRIOR_STD_Z**2]))

### No Prior

In [ ]:
# model = reconstruction.run_reconstruction(
#     sfm_dir, database, IMAGES_PATH, verbose=True,
#     options={
#         "image_names": recon_names,
#         "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#         "ba_global_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#     })

In [ ]:
# p = outputs / 'no_prior'
# p.mkdir(parents=True, exist_ok=True)
# model.write(p)

### Prior

In [ ]:
# priors only constrain global BA, and caspar doesn't support them
prior_options = {
    "image_names": recon_names,
    "use_prior_position": True,
    # "use_robust_loss_on_prior_position": True,
    # Speed options
    # "ba_use_gpu": True,
    # "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
    # "ba_global_backend": pycolmap.BundleAdjustmentBackend.CERES,
    # "ba_global_frames_ratio": 1.3,
    # "ba_global_points_ratio": 1.3,
    # "ba_local_max_num_iterations": 12,
    # "ba_local_max_refinements": 2,
    # "ba_global_max_num_iterations": 30,
    # "ba_global_max_refinements": 3,
    # "mapper": {"ba_global_ignore_redundant_points3D": True},
}
if RESUME:
    shutil.rmtree(sfm_prior_dir, ignore_errors=True)
    sfm_prior_dir.mkdir(parents=True)
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(
            database, IMAGES_PATH, sfm_prior_dir,
            options={**prior_options, "fix_existing_frames": False},
            input_path=str(outputs / 'prior'))
    model_prior = recs[0]
else:
    model_prior = reconstruction.run_reconstruction(
        sfm_prior_dir, database, IMAGES_PATH, verbose=True, options=prior_options)

In [ ]:
p = outputs / 'prior'
p.mkdir(parents=True, exist_ok=True)
model_prior.write(p)

# RTK Anchoring

Anchor the map to robobuggy RTK — final validated recipe: robo left-eye frames register against the VIRB map, carry per-anchor EKF covariances (horizontal **and** vertical), then a smooth prewarp + full retriangulation jumps the map into the prior-consistent basin before a converged joint BA.

Validated: held-out info-weighted **0.29 m** (median anchor ≈1.2σ of its RTK claim, the nominal value); road z within ±0.1 m of the USGS LiDAR DEM. Tested & rejected: RS rectification, CLAHE, stereo rigs (rights pruned — `apply_rig_config` renumbers frames and desyncs any db resumed against an existing model), anisotropic along-track covariance (the ~50 ms in-bag stamp jitter is real but modeling it per-anchor discards useful signal), robust prior loss during mapping (saturates while the model is far → priors ignored). Reprojection errors are correlated (vibration/RS/blur), so priors run at σ/`PRIOR_SCALE` — the scale was validated on held-out anchors, not fitted to training residuals. Raw z-rmse vs RTK is misleading (per-day RTK z biases): judge vertical accuracy against the DEM.

In [ ]:
ROBO_PATH = next(p for p in [Path('/kaggle/input/datasets/yu5uf5/buggy-robo'),
                             Path('/kaggle/input/buggy-robo')] if p.exists())
rtk_work = Path('/tmp/rtk')
rtk_out = Path('/kaggle/working/rtk')

CAM_FROM_GNSS2 = np.array([0.05, 0.0, -0.01])  # m, body frame; camera a few cm ahead of the antenna
ROBO_TIME_OFFSET_MS = -8.0     # nominal mid-readout+exposure lead for runs without a measured dt
# measured per-run offsets (dt grid-fit; ZED verified independently by paint-line crossings,
# and includes the directly-measured 25 ms H.264 decoder reorder lag)
TIME_OFFSETS.update({'2026-03-21_sc_0822': 60.0, '2025-10-04_4_sc_lgap': -25.0,
                     '2025-10-04_2_sc_whobaat': 0.0})
ROBO_PRIOR_STD_FLOOR = 0.08    # m, horizontal
ROBO_READOUT_S = 0.010         # rolling-shutter prior inflation: var += (v*t/2)^2
ROBO_PRIOR_STD_Z_FLOOR = 0.08  # m; z uses the per-anchor EKF variance, never a flat sigma
VIRB_PRIOR_STD_XY, VIRB_PRIOR_STD_Z = 3.0, 2.0  # loosened; RTK is the georeference
PRIOR_SCALE = 8.0  # solver-side sigma/8 on robo priors: correlated reprojection noise
                   # overweights vision. Swept 2/3/5/8 with held-out gating (held-out
                   # info-weighted 0.32/0.29/0.26/0.23) AND truth-sensitive checks:
                   # sigma/8 improved field smoothness, cross-day relative consistency,
                   # and the LiDAR-DEM z profile — i.e. its gains are real, not fitted

In [ ]:
robo = {}
for pf in sorted(ROBO_PATH.glob('*/poses.json')):
    with open(pf) as f:
        robo[pf.parent.name] = {'dir': pf.parent, **json.load(f)}


def quat_rotate(q, v):
    # q (n,4) xyzw, v (3,) -> (n,3)
    qv, w = q[:, :3], q[:, 3:]
    t = 2 * np.cross(qv, v)
    return v + w * t + np.cross(qv, t)


def ecef_to_enu_rot(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    sla, cla, slo, clo = np.sin(la), np.cos(la), np.sin(lo), np.cos(lo)
    return np.array([[-slo, clo, 0.0],
                     [-sla * clo, -sla * slo, cla],
                     [cla * clo, cla * slo, sla]])


def robo_cam_enu(run):
    """(ts_ns on the frame timeline, camera enu, horizontal var, vertical var, speed)."""
    r = robo[run]
    e = r['ekf']
    dt_ms = TIME_OFFSETS.get(run, ROBO_TIME_OFFSET_MS)
    ts = (np.array(e['t'], float) * 1000 - dt_ms) * 1e6
    enu = np.array(gt.ellipsoid_to_enu(list(np.stack([e['lat'], e['lon'], e['alt']], 1)),
                                       LAT0, LON0, ALT0))
    # ekf pose is of imu_link; move to the camera with the body lever arm
    lever = np.array(r['tf']['imu_link->gnss_2_antenna_link']) + CAM_FROM_GNSS2
    enu += quat_rotate(np.array(e['quat']), lever) @ ecef_to_enu_rot(LAT0, LON0).T
    from scipy.ndimage import maximum_filter1d
    # rolling max so brief covariance spikes aren't understated between samples
    var = maximum_filter1d(np.array(e['pos_var'])[:, :2].mean(1), size=5)
    var = np.maximum(var, ROBO_PRIOR_STD_FLOOR**2)
    var_z = maximum_filter1d(np.array(e['pos_var'])[:, 2], size=5)
    var_z = np.maximum(var_z, ROBO_PRIOR_STD_Z_FLOOR**2)
    spd = np.array(e['speed'])
    var = var + (spd * ROBO_READOUT_S / 2)**2
    return ts, enu, var, var_z, spd

def robo_roll_windows(run, v_on=1.2, bridge_s=8.0, pad_s=3.0, min_seg_m=50.0):
    """[(t0_ns, t1_ns)] moving segments of the roll — drops staging / parked / walk-back.
    Multiple windows survive (e.g. out-and-back passes); tiny segments don't."""
    e = robo[run]['ekf']
    t = np.array(e['t'], float)
    v = np.array(e['speed'], float)
    k = max(1, int(2 / np.median(np.diff(t))))
    vs = np.convolve(v, np.ones(k) / k, 'same')
    idx = np.flatnonzero(vs > v_on)
    segs = np.split(idx, np.flatnonzero(np.diff(t[idx]) > bridge_s) + 1)
    dist = np.concatenate([[0], np.cumsum(np.diff(t) * (v[1:] + v[:-1]) / 2)])
    return [((t[s[0]] - pad_s) * 1e9, (t[s[-1]] + pad_s) * 1e9)
            for s in segs if dist[s[-1]] - dist[s[0]] >= min_seg_m]


print(sorted(robo))

In [ ]:
import av
import base64
from mcap.reader import make_reader


def export_svo_frames(run):
    r = robo[run]
    out = {s: IMAGES_PATH / run / s for s in ('left', 'right')}
    for d_ in out.values():
        d_.mkdir(parents=True, exist_ok=True)
    off_ns = int(round(r['sync']['offset_s'] * 1e9))
    t0, t1 = int(r['ekf']['t'][0] * 1e9), int(r['ekf']['t'][-1] * 1e9)
    codec = av.CodecContext.create('h264', 'r')
    scores, zed_ts = {}, {}
    gyro_t, gyro_w = [], []
    with open(r['dir'] / 'vid.svo2', 'rb') as f:
        reader = make_reader(f)
        with tqdm(total=r['camera']['num_frames'], desc=run, leave=False) as progress:
            for _, ch, msg in reader.iter_messages():
                if ch.topic.endswith('/sensors'):
                    buf = base64.b64decode(json.loads(msg.data)['data'])
                    gyro_t.append(int.from_bytes(buf[16:24], 'little'))
                    gyro_w.append(float(np.linalg.norm(np.frombuffer(buf[88:100], '<f4'))))
                    continue
                if not ch.topic.endswith('/side_by_side'):
                    continue
                progress.update(1)
                try:
                    frames = [fr for pkt in codec.parse(bytes(msg.data[8:]))
                              for fr in codec.decode(pkt)]
                except av.error.InvalidDataError:  # corrupt message; decode recovers at next keyframe
                    codec = av.CodecContext.create('h264', 'r')
                    continue
                for fr in frames:
                    ts_ns = msg.log_time + off_ns
                    if not (t0 <= ts_ns <= t1):
                        continue
                    img = fr.to_ndarray(format='bgr24')
                    w = img.shape[1] // 2
                    cv2.imwrite(str(out['left'] / f'{ts_ns}.jpg'), img[:, :w])
                    cv2.imwrite(str(out['right'] / f'{ts_ns}.jpg'), img[:, w:])
                    scores[ts_ns] = sharpness_score(img[:, :w])
                    zed_ts[ts_ns] = msg.log_time
    gyro_t = np.array(gyro_t, float)
    gyro_w = np.array(gyro_w)
    omega = {ts: float(gyro_w[m].mean()) if (m := np.abs(gyro_t - zt) < 15e6).any() else 0.0
             for ts, zt in zed_ts.items()}
    return scores, omega


def export_bag_frames(run):
    """In-bag jpeg camera: R/B channels are swapped and rare frames carry burned-in overlays."""
    from rosbags.highlevel import AnyReader
    r = robo[run]
    out = IMAGES_PATH / run / 'left'
    out.mkdir(parents=True, exist_ok=True)
    t0, t1 = int(r['ekf']['t'][0] * 1e9), int(r['ekf']['t'][-1] * 1e9)
    scores = {}
    imu_t, imu_w = [], []
    dropped = 0
    with AnyReader([r['dir'] / 'data.mcap']) as reader:
        conns = [c for c in reader.connections if c.topic in (r['camera']['topic'], '/imu/data')]
        for conn, ts, raw in tqdm(reader.messages(connections=conns), desc=run, leave=False):
            m = reader.deserialize(raw, conn.msgtype)
            if conn.topic == '/imu/data':
                w = m.angular_velocity
                imu_t.append(ts)
                imu_w.append(np.degrees((w.x**2 + w.y**2 + w.z**2) ** 0.5))
                continue
            if not (t0 <= ts <= t1):
                continue
            img = cv2.imdecode(np.frombuffer(m.data, np.uint8), cv2.IMREAD_COLOR)[:, :, ::-1]
            b, g, rr = img[:, :, 0].astype(int), img[:, :, 1].astype(int), img[:, :, 2].astype(int)
            if ((np.maximum(np.maximum(b, g), rr) >= 250) &
                    (np.minimum(np.minimum(b, g), rr) <= 60)).sum() > 500:
                dropped += 1
                continue
            cv2.imwrite(str(out / f'{ts}.jpg'), img)
            scores[ts] = sharpness_score(img)
    imu_t = np.array(imu_t, float)
    imu_w = np.array(imu_w)
    omega = {ts: float(imu_w[m].mean()) if (m := np.abs(imu_t - ts) < 60e6).any() else 0.0
             for ts in scores}
    print(f'{run}: dropped {dropped} overlay frames')
    return scores, omega


def export_robo_frames(run):
    out_l = IMAGES_PATH / run / 'left'
    if (out_l / 'sharpness.json').exists():
        return
    if robo[run]['camera'].get('source', 'svo2') == 'bag_jpeg':
        scores, omega = export_bag_frames(run)
    else:
        scores, omega = export_svo_frames(run)
    with open(out_l / 'sharpness.json', 'w') as f:
        json.dump(scores, f)
    with open(out_l / 'omega.json', 'w') as f:
        json.dump(omega, f)
    print(f'{run}: saved {len(scores)} frames')


for run in robo:
    export_robo_frames(run)

In [ ]:
def select_robo_frames(run, delta_s):
    """Distance-uniform windows; sharp-enough frame with the calmest gyro per window."""
    img_dir = IMAGES_PATH / run / 'left'
    image_paths = sorted(img_dir.glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    ts_p, enu_src, var, _, spd = robo_cam_enu(run)
    sel_ts = []
    for w0, w1 in robo_roll_windows(run):
        m = (ts_p >= max(image_ts[0], w0)) & (ts_p <= min(image_ts[-1], w1))
        if m.sum() < 2:
            continue
        ts_w, v_w = ts_p[m], spd[m]
        dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
        sel_ts.extend(np.interp(np.arange(0.0, dist[-1], delta_s), dist, ts_w))
    sel_ts = np.array(sel_ts)

    with open(img_dir / 'sharpness.json') as f:
        scores = {int(k): v for k, v in json.load(f).items()}
    with open(img_dir / 'omega.json') as f:
        omega = {int(k): v for k, v in json.load(f).items()}
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2], (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
            continue
        cand = image_ts[i0:i1]
        sh = np.array([scores.get(int(u), 0.0) for u in cand])
        om = np.array([omega.get(int(u), np.inf) for u in cand])
        vv = np.interp(cand, ts_p, var)
        ok = sh >= np.percentile(sh, 60)
        ok &= vv <= 4 * vv.min()  # covariance spikes lose to much-cleaner neighbors
        if ok.any():
            om[~ok] = np.inf
            idx.append(i0 + int(np.argmin(om)))
        else:
            idx.append(i0 + int(np.argmax(sh)))
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, ts_p, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    print(f'{run}: {len(idx)} frames selected')
    return {'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
            'ts': ts, 'enu': enu, 'heading': np.arctan2(grad[:, 1], grad[:, 0])}


robo_sel = {run: select_robo_frames(run, DELTA_S) for run in robo}

# map frame positions/headings from the model; drop robo frames facing against the course
rtk_map_model = DATA_PATH / 'prior_all'
map_model = pycolmap.Reconstruction(str(rtk_map_model))
by_run = defaultdict(list)
for i in map_model.reg_image_ids():
    im = map_model.images[i]
    p = Path(im.name)
    by_run[p.parts[0]].append((int(p.stem), im.name, im.projection_center()))
map_sel = {}
for run, items in by_run.items():
    items.sort()
    enu = np.array([c for _, _, c in items])
    grad = np.gradient(enu[:, :2], axis=0)
    map_sel[run] = {'names': [n for _, n, _ in items], 'enu': enu,
                    'heading': np.arctan2(grad[:, 1], grad[:, 0])}

mm_enu = np.vstack([m['enu'][:, :2] for m in map_sel.values()])
mm_head = np.concatenate([m['heading'] for m in map_sel.values()])
tree = cKDTree(mm_enu)
for run, s in robo_sel.items():
    d, j = tree.query(s['enu'][:, :2], k=1)
    dh = np.abs(mm_head[j] - s['heading'])
    dh = np.minimum(dh, 2 * np.pi - dh)
    ok = (d < 8.0) & (dh < np.pi / 2)
    robo_sel[run] = {'names': [n for n, o in zip(s['names'], ok) if o],
                     'ts': s['ts'][ok], 'enu': s['enu'][ok], 'heading': s['heading'][ok]}
    print(f'{run}: kept {ok.sum()}/{len(ok)} course-direction frames')

robo_names = {run: {'left': s['names']} for run, s in robo_sel.items()}
# left eyes only: stereo rigs were tested and dropped (see section notes)

In [ ]:
rtk_work.mkdir(parents=True, exist_ok=True)
rtk_db = rtk_work / 'database.db'
if not rtk_db.exists():
    shutil.copy(DATA_PATH / 'database.db', rtk_db)
stage_models()


def zed_conf(run, section):
    cur, vals = None, {}
    for line in robo[run]['camera']['factory_calibration_conf'].splitlines():
        line = line.strip()
        if line.startswith('['):
            cur = line.strip('[]')
        elif '=' in line and cur == section:
            k, v = line.split('=')
            vals[k] = float(v)
    return vals


BAG_CAM_SEED = [1088.0, 640.0, 360.0, 0.0]  # unknown intrinsics; refined during phase 1

for run in robo_sel:
    if robo[run]['camera'].get('source', 'svo2') == 'bag_jpeg':
        extract_image_features(rtk_db, robo_names[run]['left'], masks=False,
                               camera_params=BAG_CAM_SEED)
    else:
        v = zed_conf(run, 'LEFT_CAM_HD')
        extract_image_features(rtk_db, robo_names[run]['left'], masks=False,
                               camera_params=[(v['fx'] + v['fy']) / 2, v['cx'], v['cy'], v['k1']])

In [ ]:
import sqlite3

# replace the mapping-era racebox priors: loose racebox on VIRB frames, tight RTK on robo left
con = sqlite3.connect(rtk_db)
con.execute('DELETE FROM pose_priors')
con.commit()
con.close()

data = load_vid_imu(DATA_PATH / 'vid_imu')
virb_enu = {run: gps_enu(d, 'racebox_gps') for run, d in data.items()}
write_pose_priors(rtk_db, virb_enu,
                  np.diag([VIRB_PRIOR_STD_XY**2, VIRB_PRIOR_STD_XY**2, VIRB_PRIOR_STD_Z**2]))

# optional drop list: torn frames (split-motion tear detector, > 30 px) and turnaround
# tails simply get no prior — their features still contribute, their positions don't
DROP_NAMES = set()
tear_file = rtk_work / 'tear_scan.json'
if tear_file.exists():
    with open(tear_file) as f:
        DROP_NAMES = {n for n, v in json.load(f).items() if v['split'] > 30}

with pycolmap.Database.open(str(rtk_db)) as db:
    name_to_img = {im.name: im for im in db.read_all_images()}
    n = 0
    for run, s in robo_sel.items():
        ts_p, enu_src, var, var_z, _ = robo_cam_enu(run)
        for name, t in zip(s['names'], s['ts']):
            if name in DROP_NAMES:
                continue
            prior = pycolmap.PosePrior()
            prior.corr_data_id = name_to_img[name].data_id
            prior.position = np.array([np.interp(t, ts_p, enu_src[:, i]) for i in range(3)])
            v = float(np.interp(t, ts_p, var))
            vz = float(np.interp(t, ts_p, var_z))
            prior.position_covariance = np.diag([v, v, vz])
            prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
            db.write_pose_prior(prior)
            n += 1
    print('robo rtk priors:', n, f'({len(DROP_NAMES)} dropped)')

In [ ]:
pairs = set()
for run, s in robo_sel.items():
    pairs |= sequential_pairs(s['names'], SEQ_K)
    for m in map_sel.values():
        pairs |= cross_pairs(s, m, CROSS_K, CROSS_R, HEADING_MAX_DEG)
for a, b in combinations(sorted(robo_sel), 2):
    pairs |= cross_pairs(robo_sel[a], robo_sel[b], CROSS_K, CROSS_R, HEADING_MAX_DEG)

(rtk_work / 'pairs.txt').write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(len(pairs), 'pairs')
match_pairs(rtk_db, rtk_work / 'pairs.txt')

# checkpoint the matched db (features + matches + priors) for BA experiments
ck = rtk_out / 'rtk_ckpt'
ck.mkdir(parents=True, exist_ok=True)
shutil.copy(rtk_db, ck / 'database.db')
with open(ck / 'robo_sel.json', 'w') as f:
    json.dump({run: {'names': s['names'], 'ts': [int(t) for t in s['ts']],
                     'enu': s['enu'].tolist(), 'heading': s['heading'].tolist()}
               for run, s in robo_sel.items()}, f)
shutil.copy(rtk_work / 'pairs.txt', ck / 'pairs.txt')

In [ ]:
# final pipeline: (1) register robo on the frozen map — priors OFF so the CASPAR GPU
# backend can run, the map itself anchors registration; (2) short direct prior BA pulls
# the field toward the anchors; (3) smooth prewarp of all frames onto the anchors, then
# FULL retriangulation (a nonrigid warp breaks rigid geometry — structure must be
# rebuilt, never kept); (4) thin the weakest tracks; (5) converged prior BA — from this
# basin it converges in a handful of iterations
base_opt = {'ba_use_gpu': True, 'ba_refine_sensor_from_rig': False,
            'ba_local_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
            'ba_global_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
            'ba_global_frames_ratio': 1.3, 'ba_global_points_ratio': 1.3,
            'ba_global_max_num_iterations': 30, 'ba_global_max_refinements': 2,
            'multiple_models': False, 'fix_existing_frames': True,
            'mapper': {'ba_global_ignore_redundant_points3D': True, 'max_reg_trials': 2}}

rtk_out.mkdir(parents=True, exist_ok=True)
p1 = rtk_out / 'phase1'
p1.mkdir(exist_ok=True)
with pycolmap.ostream():
    recs = pycolmap.incremental_mapping(rtk_db, IMAGES_PATH, p1, options=base_opt,
                                        input_path=str(rtk_map_model))
rtk_model = recs[0]
reg1 = {rtk_model.images[i].name for i in rtk_model.reg_image_ids()}
for run, s in robo_sel.items():
    print(f"{run}: registered {len(reg1 & set(s['names']))}/{len(s['names'])}")


def prior_ba(model, iterations, tol=None):
    """Direct pose-prior BA (bypasses the mapping pipeline). Robo priors scaled to
    sigma/PRIOR_SCALE; intrinsics frozen (calibrated in phase 1 — freeing them adds a
    scale/z drift mode). Priors with covariance ignore prior_position_fallback_stddev."""
    with pycolmap.Database.open(str(rtk_db)) as db:
        priors = []
        for p in db.read_all_pose_priors():
            i = p.corr_data_id.id
            if not (model.exists_image(i) and model.images[i].has_pose):
                continue
            if not model.images[i].name.split('/')[0].isdigit():
                p.position_covariance = p.position_covariance / PRIOR_SCALE**2
            priors.append(p)
    cfg = pycolmap.BundleAdjustmentConfig()
    for i in model.reg_image_ids():
        cfg.add_image(i)
    bo = pycolmap.BundleAdjustmentOptions()
    bo.refine_focal_length = False
    bo.refine_extra_params = False
    bo.refine_principal_point = False
    so = bo.ceres.solver_options
    so.max_num_iterations = iterations
    so.num_threads = -1
    if tol is not None:
        so.function_tolerance = tol
        so.use_inner_iterations = True
    pycolmap.create_pose_prior_ceres_bundle_adjuster(
        bo, pycolmap.PosePriorBundleAdjustmentOptions(), cfg, priors, model).solve()


prior_ba(rtk_model, 40)

# prewarp: long-wavelength displacement field (camera -> prior), applied to every frame
# helps converge more cleanly
from scipy.interpolate import RBFInterpolator

with pycolmap.Database.open(str(rtk_db)) as db:
    pri_pos = {p.corr_data_id.id: np.array(p.position) for p in db.read_all_pose_priors()}
robo_left = {n for s in robo_sel.values() for n in s['names']}
aids = [i for i in rtk_model.reg_image_ids()
        if i in pri_pos and rtk_model.images[i].name in robo_left]
P = np.array([rtk_model.images[i].projection_center() for i in aids])
D = np.array([pri_pos[i] - rtk_model.images[i].projection_center() for i in aids])
rbf = RBFInterpolator(P[:, :2], D, kernel='thin_plate_spline', smoothing=5000.0,
                      neighbors=256)
cap = np.percentile(np.linalg.norm(D, axis=1), 99)


def warp(xyz):
    d = rbf(xyz[:, :2])
    nn = np.linalg.norm(d, axis=1, keepdims=True)
    return xyz + d * np.minimum(1.0, cap / np.maximum(nn, 1e-9))


for f in rtk_model.frames.values():
    if not f.has_pose():
        continue
    R = f.rig_from_world.rotation
    c = -(R.matrix().T @ f.rig_from_world.translation)
    r2 = pycolmap.Rigid3d()
    r2.rotation = R
    r2.translation = -(R.matrix() @ warp(c[None])[0])
    f.rig_from_world = r2
for pid in list(rtk_model.points3D.keys()):
    rtk_model.delete_point3D(pid)
rtk_model = pycolmap.triangulate_points(rtk_model, str(rtk_db), IMAGES_PATH,
                                        str(rtk_out / 'pw_tri'))

# thin: drop track-2 points and subsample track<=4 — the discarded observations carry
# the least geometry and the most correlated error (softens the misspecified term)
rng = np.random.default_rng(0)
drop = [pid for pid, p in rtk_model.points3D.items()
        if p.track.length() <= 2 or (p.track.length() <= 4 and rng.random() > 0.6)]
for pid in drop:
    rtk_model.delete_point3D(pid)
print(f'thinned {len(drop)} points, {len(rtk_model.points3D)} remain')

prior_ba(rtk_model, 400, tol=1e-5)

In [ ]:
rtk_out.mkdir(parents=True, exist_ok=True)
(rtk_out / 'prior_all_rtk').mkdir(exist_ok=True)
rtk_model.write(rtk_out / 'prior_all_rtk')

# covariance-aware residuals vs the UNSCALED priors — the honest metrics; a raw z rmse
# vs RTK is misleading (per-day RTK z biases), judge vertical against the LiDAR DEM
with pycolmap.Database.open(str(rtk_db)) as db:
    priors = {p.corr_data_id.id: p for p in db.read_all_pose_priors()}
robo_left = {n for s in robo_sel.values() for n in s['names']}
rows = [(rtk_model.images[i].projection_center() - priors[i].position,
         float(np.sqrt(priors[i].position_covariance[0, 0])))
        for i in rtk_model.reg_image_ids()
        if i in priors and rtk_model.images[i].name in robo_left]
if rows:
    e = np.array([r[0] for r in rows])
    sig = np.array([r[1] for r in rows])
    h = np.hypot(e[:, 0], e[:, 1])
    norm = h / sig
    iw = np.sqrt((h**2 / sig**2).sum() / (1 / sig**2).sum())
    print(f'rtk anchors n={len(rows)}: info-weighted {iw:.3f}m | norm p50 '
          f'{np.median(norm):.1f} (nominal ~1.2) | within 1/2/3 sigma '
          f'{np.mean(norm<1):.0%}/{np.mean(norm<2):.0%}/{np.mean(norm<3):.0%} | '
          f'z rmse {np.sqrt((e[:, 2]**2).mean()):.2f}m (see DEM caveat)')
    for lo, hi, tag in [(0, 0.12, 'FIXED'), (0.12, 0.35, 'mid'), (0.35, 9, 'FLOAT')]:
        msk = (sig >= lo) & (sig < hi)
        if msk.sum():
            print(f'  {tag}: n={msk.sum()} rmse {np.sqrt((h[msk]**2).mean()):.3f}m '
                  f'norm p50 {np.median(norm[msk]):.1f}')
else:
    print('rtk prior residuals: no registered robo frames')

# per-VIRB-run agreement with (offset-corrected) racebox after anchoring — racebox is a
# sigma~2-3m instrument: this only guards against multi-meter blunders
for run in sorted(data, key=int):
    ims = [(int(Path(rtk_model.images[i].name).stem), rtk_model.images[i].projection_center())
           for i in rtk_model.reg_image_ids()
           if rtk_model.images[i].name.split('/')[0] == run]
    if not ims:
        continue
    fts = np.array([t for t, _ in ims], float)
    pos = np.array([p for _, p in ims])
    rts, renu = virb_enu[run]
    r = np.stack([np.interp(fts, rts, renu[:, k]) for k in range(2)], 1)
    e = pos[:, :2] - r
    print(f'{run}: vs racebox {np.sqrt((np.hypot(*e.T)**2).mean()):.2f}m rmse, '
          f'mean ({e[:, 0].mean():+.2f},{e[:, 1].mean():+.2f})')

# Visualize

### No Prior

In [13]:
# model = pycolmap.Reconstruction(str(DATA_PATH / 'outputs' / 'no_prior'))

In [15]:
# fig = viz_3d.init_figure()
# viz_3d.plot_reconstruction(fig, model, points_rgb=True)
# fig.show()

### Prior

In [ ]:
model_prior = pycolmap.Reconstruction(str(outputs / 'prior'))

In [ ]:
with pycolmap.Database.open(str(database)) as db:
    priors = db.read_all_pose_priors()
enu = {p.corr_data_id.id: p.position for p in priors}  # already ENU
errs = np.array([model_prior.images[i].projection_center() - enu[i]
                 for i in model_prior.reg_image_ids()])
print(f"rmse vs gps: {np.sqrt((errs[:, :2] ** 2).sum(1).mean()):.2f} m horizontal, "
      f"{np.sqrt((errs[:, 2] ** 2).mean()):.2f} m vertical")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

by_run = {}
for i in sorted(model_prior.reg_image_ids()):
    by_run.setdefault(model_prior.images[i].name.split('/')[0], []).append(i)

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for k, run in enumerate(sorted(by_run, key=int)):
    ids = by_run[run]
    pri = np.array([enu[i] for i in ids])
    sol = np.array([model_prior.images[i].projection_center() for i in ids])
    names = [model_prior.images[i].name for i in ids]
    c = colors[k % len(colors)]
    seg = np.concatenate([pri[:, None, :2], sol[:, None, :2],
                          np.full((len(ids), 1, 2), np.nan)], 1).reshape(-1, 2)
    fig.add_scatter(x=seg[:, 0], y=seg[:, 1], mode='lines', hoverinfo='skip',
                    line=dict(color='lightgray', width=1),
                    legendgroup=run, showlegend=False)
    fig.add_scatter(x=pri[:, 0], y=pri[:, 1], mode='markers', text=names,
                    marker=dict(color=c, size=4),
                    name=f'{run} prior', legendgroup=run)
    fig.add_scatter(x=sol[:, 0], y=sol[:, 1], mode='markers', text=names,
                    marker=dict(color=c, size=5, symbol='x'),
                    name=f'{run} solved', legendgroup=run)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(height=700)
fig.show()

In [ ]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, model_prior, points_rgb=True)
fig.show()

# Localization

Register query runs against the fixed `prior_all` model, one run at a time. Runs on copies — the original database/model are never modified — and query frames only match the map (no pairs between localized runs). Runnable standalone: Setup → Config → here.

In [ ]:
# TODO do against better model
map_database = DATA_PATH / 'database.db'
map_model_path = DATA_PATH / 'prior_all'
loc_work = Path('/tmp/loc')  # db copies + scratch
loc_out = Path('/kaggle/working/loc')

LOC_RUNS = None        # subset of int run ids; None = all of test_vid_imu
LOC_DELTA_S = 2.0      # m between query frames
LOC_GPS = 'racebox'    # 'racebox' | 'virb': selection + pairing source (virb skips uncovered runs)
LOC_SEQ_K = SEQ_K
LOC_CROSS_K = CROSS_K  # map candidates per query frame, per map run
LOC_CROSS_R = CROSS_R
LOC_PRIORS = False     # virb position priors on query frames
LOC_PRIOR_STD_XY, LOC_PRIOR_STD_Z = 3.0, 5.0

In [ ]:
loc_videos = {v.stem: v for v in (DATA_PATH / 'test_vid').glob('*.[mM][pP]4')}
loc_data = load_vid_imu(DATA_PATH / 'test_vid_imu')
loc_runs = [r for r in sorted(loc_data, key=int)
            if LOC_RUNS is None or int(r) in LOC_RUNS]
if LOC_GPS == 'virb':
    skipped = [r for r in loc_runs if not loc_data[r]['gps_data']]
    loc_runs = [r for r in loc_runs if loc_data[r]['gps_data']]
    if skipped:
        print('no virb gps, skipping:', skipped)

for run in tqdm(loc_runs, desc='videos'):
    extract_frames(loc_videos[run], loc_data[run]['camera_start'], IMAGES_PATH / run)
loc_runs

In [ ]:
loc_sel = {}
for run in loc_runs:
    loc_sel[run], _ = select_frames(run, loc_data[run], LOC_GPS, LOC_DELTA_S)

# map frame positions/headings come from the model itself, not the mapping vid_imu
map_sel = map_selection(map_model_path)
print({run: len(s['names']) for run, s in map_sel.items()})

In [ ]:
stage_models()
link_masks([n for run in loc_runs for n in loc_sel[run]['names']])
loc_work.mkdir(parents=True, exist_ok=True)
local_map_db = loc_work / 'map_database.db'
if not local_map_db.exists():
    shutil.copy(map_database, local_map_db)  # Drive reads are slow; stage the 4.4 GB db once

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, loc_work / 'colmap.LOG.')
loc_models = {}
for run in loc_runs:
    names = loc_sel[run]['names']
    work = loc_work / run
    db = work / 'database.db'
    if db.exists():  # reuse features/matches only if the selection is unchanged
        with pycolmap.Database.open(str(db)) as dbh:
            same = {im.name for im in dbh.read_all_images()
                    if im.name.split('/')[0] == run} == set(names)
        if not same:
            shutil.rmtree(work)
    if not db.exists():
        work.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_map_db, db)
        extract_image_features(db, names)

    pairs = sequential_pairs(names, LOC_SEQ_K)
    for m in map_sel.values():
        pairs |= cross_pairs(loc_sel[run], m, LOC_CROSS_K, LOC_CROSS_R, HEADING_MAX_DEG)
    (work / 'pairs.txt').write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
    match_pairs(db, work / 'pairs.txt')

    if LOC_PRIORS:
        write_pose_priors(db, {run: gps_enu(loc_data[run], 'gps_data')},
                          np.diag([LOC_PRIOR_STD_XY**2, LOC_PRIOR_STD_XY**2, LOC_PRIOR_STD_Z**2]))

    # priors only constrain global BA, and caspar doesn't support them
    opt = {'fix_existing_frames': True, 'use_prior_position': LOC_PRIORS,
           'ba_use_gpu': True,
           'ba_local_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
           'ba_global_backend': (pycolmap.BundleAdjustmentBackend.CERES if LOC_PRIORS
                                 else pycolmap.BundleAdjustmentBackend.CASPAR)}

    shutil.rmtree(work / 'model', ignore_errors=True)
    (work / 'model').mkdir()
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(db, IMAGES_PATH, work / 'model',
                                            options=opt, input_path=str(map_model_path))
    loc_models[run] = recs[0]
    reg = {loc_models[run].images[i].name for i in loc_models[run].reg_image_ids()}
    print(f'{run}: registered {len(reg & set(names))}/{len(names)} query frames')

In [ ]:
loc_results = {}
for run, model in loc_models.items():
    names = set(loc_sel[run]['names'])
    rb_ts, rb_enu = gps_enu(loc_data[run], 'racebox_gps')
    frames = []
    for i in sorted(model.reg_image_ids()):
        im = model.images[i]
        if im.name not in names:
            continue
        ts = int(Path(im.name).stem)
        c = im.projection_center()
        rb = np.array([np.interp(ts, rb_ts, rb_enu[:, j]) for j in range(3)])
        frames.append({'ts': ts, 'enu': c.tolist(),
                       'quat_xyzw': im.cam_from_world().rotation.quat.tolist(),
                       'racebox_err': (c - rb).tolist()})
    loc_results[run] = frames

    out = loc_out / run
    out.mkdir(parents=True, exist_ok=True)
    model.write(out)
    with open(out / 'poses.json', 'w') as f:
        json.dump({'delta_s': LOC_DELTA_S, 'gps': LOC_GPS, 'priors': LOC_PRIORS,
                   'frames': frames}, f)

    errs = np.array([f['racebox_err'] for f in frames])
    h = np.hypot(errs[:, 0], errs[:, 1])
    print(f"{run}: {len(frames)}/{len(names)} frames, vs racebox "
          f"{np.sqrt((h**2).mean()):.2f}m rmse / {np.percentile(h, 90):.2f}m p90 horizontal, "
          f"{np.sqrt((errs[:, 2]**2).mean()):.2f}m rmse vertical")

In [ ]:
# reload saved localizations (skip the reconstruction cells above)
# import json
# from pathlib import Path
# loc_out = Path('../../../.././tmp/loc_out')
loc_models, loc_results = {}, {}
for p in sorted(loc_out.iterdir()):
    if (p / 'poses.json').exists():
        with open(p / 'poses.json') as f:
            loc_results[p.name] = json.load(f)['frames']
        loc_models[p.name] = pycolmap.Reconstruction(str(p))
print(sorted(loc_results))

['39']


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import numpy as np
# pio.renderers.default = "notebook_connected"

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for k, run in enumerate(sorted(loc_results, key=int)):
    frames = loc_results[run]
    enu = np.array([f['enu'] for f in frames])
    errs = np.array([f['racebox_err'] for f in frames])
    rb = enu - errs
    text = [f'{run}/{f["ts"]}: {np.hypot(*e[:2]):.2f}m' for f, e in zip(frames, errs)]
    c = colors[k % len(colors)]
    seg = np.concatenate([rb[:, None, :2], enu[:, None, :2],
                          np.full((len(frames), 1, 2), np.nan)], 1).reshape(-1, 2)
    fig.add_scatter(x=seg[:, 0], y=seg[:, 1], mode='lines', hoverinfo='skip',
                    line=dict(color='lightgray', width=1),
                    legendgroup=run, showlegend=False)
    fig.add_scatter(x=rb[:, 0], y=rb[:, 1], mode='markers', text=text,
                    marker=dict(color=c, size=4),
                    name=f'{run} racebox', legendgroup=run)
    fig.add_scatter(x=enu[:, 0], y=enu[:, 1], mode='markers', text=text,
                    marker=dict(color=c, size=5, symbol='x'),
                    name=f'{run} loc', legendgroup=run)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(height=700)
fig.show()

fig = go.Figure()
for k, run in enumerate(sorted(loc_results, key=int)):
    frames = sorted(loc_results[run], key=lambda f: f['ts'])
    ts = np.array([f['ts'] for f in frames]) / 1e9
    h = np.array([np.hypot(*f['racebox_err'][:2]) for f in frames])
    fig.add_scatter(x=ts - ts[0], y=h, mode='markers', text=[f['ts'] for f in frames],
                    marker=dict(color=colors[k % len(colors)], size=4), name=run)
fig.update_layout(height=350, xaxis_title='s', yaxis_title='horizontal error m')
fig.show()

## PnP

Per-frame localization against the fixed `rtk_base` map: match query frames to map images,
lift matches to 2D-3D through the model, solve every frame independently (LO-RANSAC +
refinement). Matching happens once per run as a generous **superset**; every narrower
strategy (map-run subset, per-run k, gating radius, match budget, gating source) is a local
filter over the matched pairs — no re-matching. Query focal length must be estimated per
run (stabilizer state shifts it ~20% and PnP absorbs focal error as an along-view position
bias): pass 1 refines focal per frame, pass 2 re-solves with the run median fixed.

Colab side: superset matching cell (chunked, checkpoints a compact query-side delta after
each chunk — survives VM reclaim; features are dumped once so keypoint indexing is stable
across sessions). Local side: delta + `rtk_base/model` are enough for the whole sweep.

In [ ]:
pnp_work = Path('/tmp/pnp')
pnp_out = Path('/kaggle/working/pnp') if Path('/kaggle').exists() else outputs / 'pnp'
# the map to localize against, <name>/{model,database.db} -- same layout and repointing
# rule as ADD_MAP below; the GPU cell needs database.db, the local cells only need model/
PNP_MAP_NAME = 'rtk_base'
PNP_MAP = DATA_PATH / PNP_MAP_NAME if (DATA_PATH / PNP_MAP_NAME / 'model').exists() \
    else pnp_work / PNP_MAP_NAME
PNP_MAP_SRC = os.environ.get('SRS_PNP_MAP')  # rclone dir holding <PNP_MAP_NAME>/

PNP_SEQ_K = 6            # sequential pairs (kept for joint-BA experiments; PnP skips them)
PNP_K_VIRB, PNP_K_ROBO = 4, 3
PNP_CROSS_R, PNP_HEAD_MAX = 12.0, 55
PNP_CHUNK = 5000         # pairs per matching chunk; delta checkpoint after each
PNP_CKPT = None          # rclone remote dir for checkpoints, e.g. 'drive:buggy_hloc/pnp'
PNP_MIN_CORR = 12

In [ ]:
MAXI = 2147483647


def obj_array(arrs):
    # np.array(arrs, dtype=object) collapses to N-D whenever the arrays share a shape
    out = np.empty(len(arrs), dtype=object)
    for i, a in enumerate(arrs):
        out[i] = a
    return out


def export_pnp_delta(db_path, sel, out_npz):
    """Query keypoints + camera + verified matches touching query images, as one npz."""
    con = sqlite3.connect(db_path)
    name_to_id = {n: i for i, n in con.execute('select image_id, name from images')}
    id_to_name = {i: n for n, i in name_to_id.items()}
    qids = {name_to_id[n] for n in sel['names'] if n in name_to_id}
    kps = {i: np.frombuffer(r[2], np.float32).reshape(r[0], r[1])
           for i in qids
           for r in con.execute('select rows, cols, data from keypoints where image_id=?', (i,))}
    qcams = {con.execute('select camera_id from images where image_id=?', (i,)).fetchone()[0]
             for i in qids}
    cams = {c: (r[0], r[1], r[2], np.frombuffer(r[3], np.float64))
            for c in qcams
            for r in con.execute('select model, width, height, params from cameras'
                                 ' where camera_id=?', (c,))}
    tvg = {}
    for pid, rows, cols, data, config in con.execute(
            'select pair_id, rows, cols, data, config from two_view_geometries where rows > 0'):
        i1, i2 = pid // MAXI, pid % MAXI
        if i1 in qids or i2 in qids:
            tvg[pid] = (i1, i2, config,
                        np.frombuffer(data, np.uint32).reshape(rows, cols) if data else
                        np.zeros((0, 2), np.uint32))
    con.close()
    np.savez_compressed(
        out_npz,
        sel_names=np.array(sel['names']), sel_enu=sel['enu'], sel_heading=sel['heading'],
        image_names_all=np.array([id_to_name[i] for i in sorted(id_to_name)]),
        image_ids_all=np.array(sorted(id_to_name)),
        kp_ids=np.array(sorted(kps)), kp_data=obj_array([kps[i] for i in sorted(kps)]),
        cam_ids=np.array(sorted(cams)),
        cam_data=np.array([(cams[c][0], cams[c][1], cams[c][2], *cams[c][3])
                           for c in sorted(cams)]),
        tvg_pairs=np.array([(v[0], v[1], v[2]) for v in tvg.values()]),
        tvg_matches=obj_array([v[3] for v in tvg.values()]),
        allow_pickle=True)
    return len(tvg)

In [ ]:
# superset matching per run (GPU). Reuses loc_sel/map_sel from the cells above.
pnp_work.mkdir(parents=True, exist_ok=True)
pnp_out.mkdir(parents=True, exist_ok=True)
ROBO_RUNS = {r for r in map_sel if not r.split('/')[0].isdigit()}

for run in loc_runs:
    sel = loc_sel[run]
    db = pnp_work / f'{run}.db'
    if not db.exists():
        shutil.copy(PNP_MAP / 'database.db', db)
        extract_image_features(db, sel['names'])
    pairs = sequential_pairs(sel['names'], PNP_SEQ_K)
    for mrun, ms in map_sel.items():
        k = PNP_K_ROBO if mrun in ROBO_RUNS else PNP_K_VIRB
        pairs |= cross_pairs(sel, ms, k, PNP_CROSS_R, PNP_HEAD_MAX)
    con = sqlite3.connect(db)
    name_to_id = {n: i for i, n in con.execute('select image_id, name from images')}
    qids = {name_to_id[n] for n in sel['names']}
    done = set()
    for pid, in con.execute('select pair_id from two_view_geometries'):
        if pid // MAXI in qids or pid % MAXI in qids:
            done.add(pid)
    con.close()

    def pid_of(a, b):
        ia, ib = sorted((name_to_id[a], name_to_id[b]))
        return ia * MAXI + ib

    todo = sorted(p for p in pairs if pid_of(*p) not in done)
    print(f'{run}: {len(pairs)} pairs, {len(todo)} to match')
    delta = pnp_out / f'delta_{run}.npz'
    for c0 in range(0, len(todo), PNP_CHUNK):
        pf = pnp_work / 'chunk.txt'
        pf.write_text('\n'.join(f'{a} {b}' for a, b in todo[c0:c0 + PNP_CHUNK]))
        match_pairs(db, pf)
        n = export_pnp_delta(db, sel, delta)
        if PNP_CKPT:
            subprocess.run(['rclone', 'copyto', str(delta), f'{PNP_CKPT}/delta_{run}.npz'])
        print(f'{run}: chunk {c0 // PNP_CHUNK + 1}/{-(-len(todo) // PNP_CHUNK)}, '
              f'{n} matched pairs checkpointed')

In [ ]:
# local: correspondence groups from delta(s) + model; strategies filter these
pnp_model = pycolmap.Reconstruction(str(PNP_MAP / 'model'))
pnp_xyz = {pid: p.xyz for pid, p in pnp_model.points3D.items()}
pnp_mid = {pnp_model.images[i].name: i for i in pnp_model.reg_image_ids()}
_p3d_cache = {}


def _map_p3d(name):
    if name not in _p3d_cache:
        mid = pnp_mid.get(name)
        if mid is None:
            _p3d_cache[name] = None
        else:
            im = pnp_model.images[mid]
            arr = np.full(len(im.points2D), -1, np.int64)
            for k, p2 in enumerate(im.points2D):
                if p2.has_point3D():
                    arr[k] = p2.point3D_id
            _p3d_cache[name] = (arr, im.projection_center())
    return _p3d_cache[name]


def load_pnp_groups(delta_paths):
    """{query name: [group]}, each group one matched map image with its 2D-3D corrs."""
    kp_by_name, sel, cam_row, seen = {}, None, None, set()
    groups = defaultdict(list)
    for dp in delta_paths:
        d = np.load(dp, allow_pickle=True)
        name_of = dict(zip(d['image_ids_all'].tolist(), d['image_names_all'].tolist()))
        for i, k in zip(d['kp_ids'].tolist(), d['kp_data']):
            kp_by_name[name_of[i]] = k
        if sel is None or len(d['sel_names']) > len(sel['names']):
            sel = {'names': list(d['sel_names']), 'enu': d['sel_enu'],
                   'heading': d['sel_heading']}
        if cam_row is None:
            cam_row = next(iter(d['cam_data']))
        qnames = set(d['sel_names'])
        for (i1, i2, cfg), matches in zip(d['tvg_pairs'], d['tvg_matches']):
            n1, n2 = name_of[int(i1)], name_of[int(i2)]
            q, mp = (n1, n2) if n1 in qnames else (n2, n1) if n2 in qnames else (None, None)
            if q is None or mp in qnames or (q, mp) in seen or not len(matches):
                continue
            seen.add((q, mp))
            got = _map_p3d(mp)
            if got is None:
                continue
            p3, mc = got
            qm = matches if n1 == q else matches[:, ::-1]
            valid = p3[qm[:, 1]] >= 0
            if not valid.any():
                continue
            groups[q].append({'mrun': mp.split('/')[0], 'mcenter': mc[:2],
                              'q_xy': kp_by_name[q][qm[valid, 0]][:, :2],
                              'pts': np.array([pnp_xyz[p3[j]] for j in qm[valid, 1]])})
    return groups, sel, cam_row


def pnp_solve(groups, sel, cam_row, cfg, stride=1):
    """cfg: runs, k, r, max_pairs, min_corr, focal ('auto'|value|None), refine_focal,
    gate positions default to sel['enu'] (racebox); pass gate_pos to override."""
    gpos = cfg.get('gate_pos', np.asarray(sel['enu'])[:, :2])
    ref = pycolmap.AbsolutePoseRefinementOptions()
    est = pycolmap.AbsolutePoseEstimationOptions()
    if cfg.get('refine_focal'):
        ref.refine_focal_length = ref.refine_extra_params = True
    cam_tpl = pycolmap.Camera(
        camera_id=1, model=pycolmap.CameraModelId(int(cam_row[0])),
        width=int(cam_row[1]), height=int(cam_row[2]), params=list(cam_row[3:]))
    if cfg.get('focal') == 'auto':
        probe = dict(cfg, focal=None, refine_focal=True)
        rows = pnp_solve(groups, sel, cam_row, probe, stride=4)
        cfg = dict(cfg, focal=float(np.median([r['focal'] for r in rows if r])),
                   k1=float(np.median([r['k1'] for r in rows if r])),
                   refine_focal=False)
        print(f"auto focal -> {cfg['focal']:.1f} k1 -> {cfg['k1']:+.4f}")
    rows = []
    for i, qname in enumerate(sel['names']):
        if i % stride:
            rows.append(None)
            continue
        gs = groups.get(qname, [])
        if cfg.get('runs') is not None:
            gs = [g for g in gs if g['mrun'] in cfg['runs']]
        ds = [float(np.hypot(*(g['mcenter'] - gpos[i]))) for g in gs]
        if cfg.get('r') is not None:
            gs, ds = zip(*[(g, dd) for g, dd in zip(gs, ds) if dd <= cfg['r']]) \
                if any(dd <= cfg['r'] for dd in ds) else ([], [])
        if cfg.get('k') is not None:
            byrun = defaultdict(list)
            for g, dd in zip(gs, ds):
                byrun[g['mrun']].append((dd, g))
            gs = [g for lst in byrun.values()
                  for _, g in sorted(lst, key=lambda x: x[0])[:cfg['k']]]
        elif cfg.get('max_pairs') is not None:
            gs = [gs[j] for j in np.argsort(ds)[:cfg['max_pairs']]]
        if not gs:
            rows.append(None)
            continue
        q_xy = np.vstack([g['q_xy'] for g in gs])
        pts = np.vstack([g['pts'] for g in gs])
        if len(q_xy) < cfg.get('min_corr', PNP_MIN_CORR):
            rows.append(None)
            continue
        camera = pycolmap.Camera(camera_id=1, model=cam_tpl.model, width=cam_tpl.width,
                                 height=cam_tpl.height, params=list(cam_tpl.params))
        if isinstance(cfg.get('focal'), float):
            camera.params = [cfg['focal'], *camera.params[1:]]
        if cfg.get('k1') is not None:
            camera.params = [*camera.params[:3], cfg['k1']]
        r = pycolmap.estimate_and_refine_absolute_pose(q_xy, pts, camera, est, ref)
        rows.append(None if r is None else
                    {'center': r['cam_from_world'].inverse().translation,
                     'inl': int(np.count_nonzero(r['inlier_mask'])),
                     'focal': float(camera.params[0]),
                     'k1': float(camera.params[3]), 'npairs': len(gs)})
    return rows

In [ ]:
def pnp_metrics(rows, sel, gps_ts, gps_enu_arr, label=''):
    """dt-refit vs the gps series, then residual stats + consecutive-frame jitter."""
    ok = [(i, r) for i, r in enumerate(rows) if r]
    idx = np.array([i for i, _ in ok])
    C = np.array([r['center'] for _, r in ok])
    ts_q = np.array([int(sel['names'][i].split('/')[1].split('.')[0]) for i in idx], float)
    best = min(((dt, float(np.median(np.hypot(
        *(C[:, :2] - np.stack([np.interp(ts_q + dt * 1e6, gps_ts, gps_enu_arr[:, j])
                               for j in range(2)], 1)).T))))
        for dt in np.arange(-1500, 1501, 10.0)), key=lambda x: x[1])
    dt = best[0]
    g = np.stack([np.interp(ts_q + dt * 1e6, gps_ts, gps_enu_arr[:, j]) for j in range(3)], 1)
    e = C - g
    h = np.hypot(e[:, 0], e[:, 1])
    dv = np.linalg.norm(np.diff(C[:, :2], axis=0) - np.diff(g[:, :2], axis=0), axis=1)
    jit = dv[np.diff(idx) == 1]
    out = {'loc': len(ok), 'total': len(rows), 'dt_ms': dt,
           'p50': float(np.median(h)), 'p90': float(np.percentile(h, 90)),
           'rmse': float(np.sqrt((h ** 2).mean())),
           'z_mean': float(e[:, 2].mean()),
           'inl_p50': int(np.median([r['inl'] for _, r in ok])),
           'jit_p50': float(np.median(jit)), 'jit_p90': float(np.percentile(jit, 90))}
    print(f"{label:26} loc {out['loc']}/{out['total']} dt {dt:+5.0f}ms | "
          f"p50 {out['p50']:.3f} p90 {out['p90']:.3f} rmse {out['rmse']:.3f} | "
          f"z {out['z_mean']:+.2f} inl {out['inl_p50']} | "
          f"jit {out['jit_p50']:.3f}/{out['jit_p90']:.3f}")
    return out

In [ ]:
VIRB_MAP_RUNS = sorted(r for r in map_sel if r.split('/')[0].isdigit())
ROBO_MAP_RUNS = sorted(r for r in map_sel if not r.split('/')[0].isdigit())
PNP_STRATEGIES = {
    'virb_k2_r6': {'runs': VIRB_MAP_RUNS, 'k': 2, 'r': 6.0, 'focal': 'auto'},
    'virb_k4_r12': {'runs': VIRB_MAP_RUNS, 'k': 4, 'r': 12.0, 'focal': 'auto'},
    'all_k2_r6': {'k': 2, 'r': 6.0, 'focal': 'auto'},
    'all_k4_r12': {'k': 4, 'r': 12.0, 'focal': 'auto'},
    'robo_k3_r12': {'runs': ROBO_MAP_RUNS, 'k': 3, 'r': 12.0, 'focal': 'auto'},
    'budget4': {'max_pairs': 4, 'r': 12.0, 'focal': 'auto'},
    'budget16': {'max_pairs': 16, 'r': 12.0, 'focal': 'auto'},
}

pnp_results = {}
for run in loc_runs:
    deltas = sorted(pnp_out.glob(f'delta*{run}*.npz'))
    if not deltas:
        continue
    groups, sel, cam_row = load_pnp_groups(deltas)
    gps_ts, gps_enu_arr = gps_enu(loc_data[run], 'racebox_gps')
    print(f'=== {run}: {sum(len(v) for v in groups.values())} matched pair-groups ===')
    for name, cfg in PNP_STRATEGIES.items():
        rows = pnp_solve(groups, sel, cam_row, dict(cfg))
        pnp_results[(run, name)] = (rows, pnp_metrics(rows, sel, gps_ts, gps_enu_arr, name))

In [ ]:
# review overlay: gps track vs per-strategy PnP centers, colored by inlier count
import plotly.graph_objects as go

run = loc_runs[0]
fig = go.Figure()
gps_ts, gps_enu_arr = gps_enu(loc_data[run], 'racebox_gps')
fig.add_trace(go.Scatter(x=gps_enu_arr[:, 0], y=gps_enu_arr[:, 1], mode='lines',
                         name='racebox', line=dict(color='gray', width=1)))
for name in PNP_STRATEGIES:
    rows, _ = pnp_results.get((run, name), (None, None))
    if not rows:
        continue
    C = np.array([r['center'] for r in rows if r])
    inl = [r['inl'] for r in rows if r]
    fig.add_trace(go.Scatter(x=C[:, 0], y=C[:, 1], mode='markers', name=name,
                             marker=dict(size=4, color=inl, colorscale='Viridis'),
                             visible='legendonly' if name != 'all_k2_r6' else True))
fig.update_layout(height=700, yaxis_scaleanchor='x')
fig.show()

# Add Runs

Match two autumn-foliage rolls — 982 (2023-09-30) and 1005 (2023-10-28) — against
the map in `ADD_MAP` (`rtk_spring` when this was run) so a later step can add them to the map as a foliage layer. The map has no
September/October VIRB foliage, and these two are the best-conditioned autumn rolls in the
season probe (98%/97% localized, inl_p50 1454/1770, pitch +4.7°/+2.8°).

**Produces matches only.** No bundle adjustment, no reconstruction, no write to the map.

Pairs are built in three classes, sized for BA rather than for localization:

- **run→map**, gated on position + heading and stratified by map *source run* (`ADD_K_MAP` per
  run, not the k nearest overall). The k nearest are usually all from one run; tying a new roll
  to a single run lets that run's error propagate straight into it.
- **intra-run**, at skip offsets `ADD_SKIPS`. Consecutive-only pairs triangulate every new
  point at ~4°; offset 12 at 2 m spacing gives ~40° on 30 m structure.
- **982↔1005**, gated the same way. Two independent foliage sources matched directly, so
  neither is a lone bridge for future foliage runs and their later disagreement is measurable.

Gating is geometric (position + heading), not NetVLAD: retrieval is the weaker signal across
seasons. The gate radius is set by the recall check below, against the map images the season
probe actually matched.

## Where the initial poses come from

Every new frame needs a position before it has a pose: one to aim its gate from, and one for
registration to start at. `ADD_POSES` chooses where that comes from.

1. **GPS priors** (`ADD_POSES` unset) — the roll's own VIRB track. The simple path, what the
   original 982/1005 addition ran, and adequate wherever the track is good.
2. **Recommended: solve once, bridge the poor sections, then redo the model with those
   initialisations.** Register the new runs normally, find the stretches where localisation
   came out degraded, reconstruct camera poses across those gaps from the confident poses
   bracketing them, then rebuild — using the corrected poses both to re-aim retrieval and to
   initialise. This is what produced the current `rtk_base`. `tmp/bridge/` is the bridging
   method; `tmp/addrun2/` is the harness that redid the model with its output
   (`scripts/a1_poses.py` builds the pose table, `scripts/a2_aim.py` re-aims the gate).

The file is one npz per roll at `<ADD_POSES>/<roll>.npz`, keyed by frame name: `names` plus a
per-frame `centre` (`position` / `aim_enu` also read), optionally `sigma_along_m`, `section`,
`method` and `trust` — the format `tmp/bridge/outputs/corrected_{982,1005}.npz` and
`tmp/addrun2/outputs/` are written in. Frames it does not name keep their GPS position, so a
file covering only the corrected sections is a valid input, and `ADD_POSE_TRUST` filters the
rest (that is how the bridge's `do_not_adopt` section stays out). The rebuild's own table was
the solved `rtk_base` pose for all 1,324 frames with 136 bridged poses substituted, so the
aiming is model-quality everywhere, not just where a correction existed.

Worth the extra pass — 982/1005 rebuilt into `rtk_spring`, against the GPS-aimed `rtk_base`:

| | GPS-aimed (`rtk_base`) | re-aimed + bridged |
|---|---|---|
| 982 anchored cross-run support, arc 342–498 m | 0.063 | **0.333** |
| whole-map RTK horizontal, info-weighted | 0.2259 m | **0.1718 m** |
| decay-removed energy swing over that stretch | 34.2 J/kg | **11.6** (4.4 after iterated retriangulation/BA) |

A coasting buggy's energy cannot rise, so a 34.2 J/kg swing is the *map* deformed, not the
roll. That stretch also had the most observations per frame and the lowest reprojection error
in the whole map: every conventional metric called it the best region, and 74% of each frame's
observations were private to 982.

Every frame of a new run then carries a prior row at its initialisation pose — the corrected
pose where there is one, the GPS or model pose everywhere else — all at the same **weak**
`ADD_PRIOR_SIGMA_H` / `ADD_PRIOR_SIGMA_Z` = 3.0 / 10.0 m. The uniform σ is the point: the row
regularises the run towards where it was initialised, it does not rank frames by how well
their pose is known, and no frame is left without one. 3.0 m is already 7.7× looser than the
bridge's median measured `sigma_along_m`, 73× at its tightest frame. Aiming and initialisation
are the corrected poses' job, not constraining the solve, and the solve agreed: the solved
poses landed 0.49σ (982) and 0.21σ (1005) from them, so the matches decided the answer.

The rebuild also raised `ADD_K_MAP` 2 → 3 and skipped pairs the first run had already
attempted (attempted-but-empty counts as done) — both in `tmp/addrun2/scripts/a2_aim.py`.

Runnable standalone: Setup → Config → here.

In [ ]:
ADD_ROLLS = ['982', '1005']
add_work = Path('/tmp/add')
add_out = Path('/kaggle/working/add') if Path('/kaggle').exists() else outputs / 'add'
add_out.mkdir(parents=True, exist_ok=True)
# per run: <roll>.MP4, <roll>.json (vid_imu schema), <roll>.png (mask)
ADD_PATH = DATA_PATH / 'add' if (DATA_PATH / 'add').exists() else add_work / 'input'
# the map being added to -- the one constant to repoint. rtk_base is the current best map
# (982+1005 joint model, db carrying their features and matches), so a new addition builds
# on it. rtk_spring is the original leaf-off map: use it to reproduce the 982/1005 addition,
# which rtk_base already contains. Both are <name>/{model,database.db}.
ADD_MAP_NAME = 'rtk_base'
ADD_MAP = DATA_PATH / ADD_MAP_NAME if (DATA_PATH / ADD_MAP_NAME / 'model').exists() \
    else add_work / ADD_MAP_NAME
ADD_MAP_DB = ADD_MAP / 'database.db'
# remote-shaped config stays in the environment; nothing about it is checked in
ADD_SRC = os.environ.get('SRS_ADDMATCH_SRC')     # rclone dir holding the run inputs
ADD_MAP_SRC = os.environ.get('SRS_ADDMATCH_MAP')  # rclone dir holding <ADD_MAP_NAME>/model
ADD_CKPT = os.environ.get('SRS_ADDMATCH_CKPT')   # rclone dir for chunk checkpoints

# initial poses for the new frames: what the retrieval gate is aimed from and what
# registration starts at. One npz per roll at <ADD_POSES>/<roll>.npz, keyed by frame name:
# names + centre (position/aim_enu also read), optionally sigma_along_m, section, trust.
# Frames the file does not name keep their gps position, so a file covering only the
# corrected sections is a valid input. Unset = the gps path, what the original addition ran.
ADD_POSES = os.environ.get('SRS_ADD_POSES')
ADD_POSE_TRUST = ('use', 'marginal')   # per-frame trust values adopted from the file
# weak on purpose, and identical on every frame of the run: these poses aim and initialise,
# they do not constrain the solve, and the row is not a per-frame confidence. 3.0 m is
# what every VIRB run in the map entered at; 10 m vertically because a z inherited from the
# map must not be fed back to it as an observation
ADD_PRIOR_SIGMA_H, ADD_PRIOR_SIGMA_Z = 3.0, 10.0

ADD_DELTA_S = 2.0                 # m between frames, matching the map's own runs
ADD_SKIPS = (1, 2, 3, 5, 8, 12)   # intra-run offsets -> 2-24 m baselines
ADD_K_MAP = 2                     # map candidates per map source run (this is the stratification)
ADD_K_PAIR = 6                    # 982 <-> 1005 candidates per frame
ADD_R, ADD_HEAD_MAX = 25.0, 60    # m / deg gate; R chosen by the recall check below
ADD_CHUNK = 500                   # pairs per chunk; a reclaim costs at most one chunk
ADD_RATE = 6.14                   # pairs/s, conservative floor for the estimate; measured 12.9 on 2xT4
ADD_MAX_GPU_MIN = 180             # hard budget guard: stop rather than spend more

# dt grid-fit of the season probe's solved poses against the virb gps stream (preflight cell);
# same convention as TIME_OFFSETS — gps interpolated at frame_ts + dt
ADD_TIME_OFFSETS = {'982': 830.0, '1005': 270.0}
# season probe two-pass per-run intrinsics (era_probe/season/results.json)
ADD_INTRINSICS = {'982': (763.09, -0.0212), '1005': (772.18, -0.0353)}

In [ ]:
# local prep (needs the srs data dir): stage each new run as a vid_imu-schema input so the
# GPU cells below use the same selection/pairing code as every other run.
SRS_DATA = Path(os.environ.get('SRS_DATA_PATH', '../../../../data'))
VIRB_VIDEO_START_OFFSET_MS = 130   # first frame leads the fit video_start event (smooth.ipynb)


def _srs_resolve(uri):
    for pre, sub in [('[[archive]]/', 'archive/'), ('[[gdrive]]/', 'gdrive/'),
                     ('[[videos]]/', 'videos/'), ('[[fit]]/', 'virbs/')]:
        if uri.startswith(pre):
            return SRS_DATA / (sub + uri[len(pre):])
    return SRS_DATA / uri


def _fit_gps(uri):
    """decoded gps_metadata from lib.fit's cache: 10 Hz, 1 Hz fixes + doppler dead reckoning"""
    rel = str(_srs_resolve(uri).relative_to(SRS_DATA))
    cache = SRS_DATA / 'cache' / (rel.replace('/', '_').replace('.fit', '') + '.json')
    if not cache.exists():
        raise FileNotFoundError(f'{cache} — decode the fit once via lib.fit.load_fit_file')
    with open(cache) as f:
        msgs = json.load(f)
    g = [m for m in msgs['gps_metadata_mesgs'] if abs(m.get('position_lat') or 0) > 1]
    g.sort(key=lambda m: m['timestamp'] * 1000 + m['timestamp_ms'])
    return g


def add_prep(rolls, out_dir, srs_db=None):
    import sqlite3
    out_dir.mkdir(parents=True, exist_ok=True)
    con = sqlite3.connect(srs_db or SRS_DATA / 'db/srs.db')
    for roll in rolls:
        files = {t: (fid, uri, ls) for fid, t, uri, ls in con.execute(
            'select f.id, f.type, f.uri, rf.local_start_ms from rollfile rf '
            'join file f on f.id=rf.file_id where rf.roll_id=?', (int(roll),))}
        vfid, vuri, vls = files['video_preview']
        _, furi, fls = files['fit']
        ev = {}
        for t, tag, ms in con.execute(
                'select type, tag, timestamp_ms from rollevent where roll_id=?', (int(roll),)):
            ev[f'{t}:{tag}' if t == 'hill_start' else t] = ms
        # the fit's video_start event is the video's t=0 on the fit device clock
        shift = fls - vls
        camera_start_ms = shift - VIRB_VIDEO_START_OFFSET_MS
        t0, t1 = ev['hill_start:1'] + fls, ev['roll_end'] + fls   # roll window, device ms

        g = [m for m in _fit_gps(furi)
             if t0 <= m['timestamp'] * 1000 + m['timestamp_ms'] <= t1]
        ts = [(m['timestamp'] * 1000 + m['timestamp_ms']) * 1_000_000 for m in g]
        out = {
            'roll': int(roll), 'file_id': vfid, 'uri': vuri, 'events': ev,
            'camera_start': int(camera_start_ms * 1_000_000),
            'camera_start_correction_ms': VIRB_VIDEO_START_OFFSET_MS,
            'roll_window_ns': [int(t0 * 1e6), int(t1 * 1e6)],
            'gps_source': 'virb fit gps_metadata (10 Hz: 1 Hz fixes + doppler dead reckoning)',
            'gps_data': [{'lat': m['position_lat'] / 2 ** 31 * 180,
                          'long': m['position_long'] / 2 ** 31 * 180,
                          'alt': m.get('enhanced_altitude') or 0.0, 'timestamp': t}
                         for m, t in zip(g, ts)],
            'velocity': [{'vx': m['velocity'][0], 'vy': m['velocity'][1],
                          'vz': m['velocity'][2], 'timestamp': t} for m, t in zip(g, ts)],
        }
        mask = cv2.imread(str(SRS_DATA / f'masks/{vfid}.png'), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f'no reviewed mask for file {vfid}')
        mask = np.where(mask > 127, 255, 0).astype(np.uint8)
        out['mask_frac'] = float((mask == 0).mean())
        # colmap masking is a post-filter, so raise the budget to keep ~4096 kept features
        out['max_num_features'] = int(min(6000, round(4096 / (1 - out['mask_frac']))))
        out['focal_init'], out['k1_init'] = ADD_INTRINSICS[roll]
        cv2.imwrite(str(out_dir / f'{roll}.png'), mask)
        with open(out_dir / f'{roll}.json', 'w') as f:
            json.dump(out, f)
        vid = out_dir / f'{roll}.MP4'
        if not vid.exists():
            shutil.copy(_srs_resolve(vuri), vid)
        print(f'{roll}: {len(g)} gps samples over {(t1 - t0) / 1000:.0f} s, '
              f'mask_frac {out["mask_frac"]:.4f}, budget {out["max_num_features"]}, '
              f'video {vid.stat().st_size / 1e6:.0f} MB')
    con.close()


add_prep(ADD_ROLLS, ADD_PATH)

In [ ]:
import sqlite3
import time


def add_load(path):
    """vid_imu-schema loader for the new runs; ADD_TIME_OFFSETS is applied to the virb streams
    the way load_vid_imu applies TIME_OFFSETS to racebox — shift the stream by -dt."""
    d = load_vid_imu(path)
    for run, dd in d.items():
        off = -ADD_TIME_OFFSETS.get(run, 0.0) * 1e6
        for key in ('gps_data', 'velocity'):
            for s in dd.get(key, []):
                s['timestamp'] += off
    return d


def add_heading(names, d, win_ms=1000.0):
    """select_frames differences consecutive selected positions, 2 m apart; one virb stutter
    (the 1 Hz fix injection stalling then catching up) flips that by 180 deg and the heading
    gate then drops every map candidate for that frame. Use a +-1 s baseline instead."""
    ts = np.array([int(n.split('/')[1].split('.')[0]) for n in names], float)
    g = d['gps_data']
    gts = np.array([s['timestamp'] for s in g], float)
    enu = np.array(gt.ellipsoid_to_enu([[s['lat'], s['long'], s['alt']] for s in g],
                                       LAT0, LON0, ALT0))[:, :2]
    step = np.stack([np.interp(ts + win_ms * 1e6, gts, enu[:, j])
                     - np.interp(ts - win_ms * 1e6, gts, enu[:, j]) for j in range(2)], 1)
    return np.arctan2(step[:, 1], step[:, 0])


def add_map_z(xy, map_sel):
    """virb altitude is unusable as an ENU z; take the map's own road height instead"""
    pts = np.vstack([s['enu'][:, :2] for s in map_sel.values()])
    z = np.concatenate([s['enu'][:, 2] for s in map_sel.values()])
    return z[cKDTree(pts).query(np.asarray(xy))[1]]


def add_chord_heading(enu, half=5):
    """Heading over a +-half-frame chord. np.gradient's one-step form put 982's 374.5 m splice
    frame 67 deg off, and the 60 deg gate then rejected all 192 of its map candidates."""
    e = np.asarray(enu)[:, :2]
    i = np.arange(len(e))
    step = e[np.minimum(i + half, len(e) - 1)] - e[np.maximum(i - half, 0)]
    return np.arctan2(step[:, 1], step[:, 0])


def add_apply_poses(sel, npz_path, trust=None):
    """Take frame positions from a pose file instead of from the gps track.

    The file names frames and gives each a centre (`centre` | `position` | `aim_enu`), plus
    optional `sigma_along_m` / `sigma_m`, `section` and `trust`; rows whose trust is not in
    `trust` are ignored, which is how a bridge's do_not_adopt sections stay out. Frames it
    does not name keep their gps position. Heading is then recomputed for the whole run --
    it is a different track now, and the gate uses position and heading together.

    `sigma_m` rides along into priors_<roll>.npz for provenance only -- it is not the prior
    weight and does not decide which frames get a prior row. Every frame of the run gets one,
    at whatever pose it ends up initialised from; see add_ba_pose_priors.
    """
    z = np.load(npz_path, allow_pickle=False)
    pos = next(z[k] for k in ('centre', 'position', 'aim_enu') if k in z.files)
    sig = next((z[k] for k in ('sigma_along_m', 'sigma_m') if k in z.files), None)
    idx = {n: i for i, n in enumerate(sel['names'])}
    before = sel['enu'].copy()
    hits = []
    for j, name in enumerate(str(s) for s in z['names']):
        i = idx.get(name)
        if i is None or (trust and 'trust' in z.files and str(z['trust'][j]) not in trust):
            continue
        sel['enu'][i] = pos[j]
        sel['source'][i] = 'file'
        sel['sigma_m'][i] = np.nan if sig is None else float(sig[j])
        if 'section' in z.files:
            sel['section'][i] = str(z['section'][j])
        hits.append(i)
    if hits:
        sel['heading'] = add_chord_heading(sel['enu'])
    d = np.linalg.norm(sel['enu'][hits, :2] - before[hits, :2], axis=1) if hits else None
    return {'applied': len(hits), 'named': len(z['names']), 'of': len(sel['names']),
            'moved_p50_m': round(float(np.median(d)), 2) if hits else 0.0}


def add_push(local, name):
    if ADD_CKPT:
        subprocess.run(['rclone', 'copyto', str(local), f'{ADD_CKPT}/{name}.tmp'], check=True)
        subprocess.run(['rclone', 'moveto', f'{ADD_CKPT}/{name}.tmp', f'{ADD_CKPT}/{name}'],
                       check=True)


def add_pull(name, local):
    if ADD_CKPT and not Path(local).exists():
        subprocess.run(['rclone', 'copyto', f'{ADD_CKPT}/{name}', str(local)], check=False)
    return Path(local).exists()


def add_log(msg):
    line = f'[{time.strftime("%H:%M:%S")}] {msg}'
    print(line, flush=True)
    with open(add_out / 'add.log', 'a') as f:
        f.write(line + '\n')
    if ADD_CKPT:
        subprocess.run(['rclone', 'copyto', str(add_out / 'add.log'), f'{ADD_CKPT}/add.log'],
                       check=False)


def add_save_state(state):
    (add_out / 'state.json').write_text(json.dumps(state, indent=1))
    add_push(add_out / 'state.json', 'state.json')


def export_add_features(db_path, names, out_npz):
    """raw keypoint/descriptor blobs + camera/rig/frame rows, so a fresh kernel restores
    byte-identical features (and therefore keypoint indices) into a clean copy of the map db"""
    con = sqlite3.connect(db_path)
    want = set(names)
    ids = {n: i for i, n in con.execute('select image_id, name from images') if n in want}
    rows = []
    for n, i in sorted(ids.items()):
        cam, = con.execute('select camera_id from images where image_id=?', (i,)).fetchone()
        # a frame can yield ZERO keypoints; colmap then stores rows=0 with a NULL blob
        kp = con.execute('select rows, cols, data from keypoints where image_id=?', (i,)).fetchone()
        de = con.execute('select type, rows, cols, data from descriptors where image_id=?',
                         (i,)).fetchone()
        kp = (kp[0], kp[1], kp[2] or b'') if kp else (0, 6, b'')
        de = (de[0], de[1], de[2], de[3] or b'') if de else (0, 0, 128, b'')
        fr = con.execute('select frame_id, sensor_id, sensor_type from frame_data '
                         'where data_id=? and sensor_type=0', (i,)).fetchone()
        rig = con.execute('select rig_id from frames where frame_id=?',
                          (fr[0],)).fetchone() if fr else None
        rows.append((n, i, cam, kp, de, fr, rig))
    cams = {c: con.execute('select model, width, height, params, prior_focal_length '
                           'from cameras where camera_id=?', (c,)).fetchone()
            for c in {r[2] for r in rows}}
    rigs = {r[6][0]: con.execute('select ref_sensor_id, ref_sensor_type from rigs where rig_id=?',
                                 (r[6][0],)).fetchone()
            for r in rows if r[6]}
    con.close()
    np.savez_compressed(
        out_npz,
        names=np.array([r[0] for r in rows]), image_ids=np.array([r[1] for r in rows]),
        camera_ids=np.array([r[2] for r in rows]),
        kp_shape=np.array([[r[3][0], r[3][1]] for r in rows]),
        kp_blob=obj_array([np.frombuffer(r[3][2], np.uint8) for r in rows]),
        de_shape=np.array([[r[4][0], r[4][1], r[4][2]] for r in rows]),
        de_blob=obj_array([np.frombuffer(r[4][3], np.uint8) for r in rows]),
        frame_rows=np.array([[r[5][0], r[5][1], r[5][2], r[6][0]] if r[5] else [-1, -1, -1, -1]
                             for r in rows]),
        cam_ids=np.array(sorted(cams)),
        cam_rows=np.array([[cams[c][0], cams[c][1], cams[c][2], cams[c][4]] for c in sorted(cams)]),
        cam_params=obj_array([np.frombuffer(cams[c][3], np.uint8) for c in sorted(cams)]),
        rig_ids=np.array(sorted(rigs)),
        rig_rows=np.array([[rigs[g][0], rigs[g][1]] for g in sorted(rigs)]),
        allow_pickle=True)


def restore_add_features(db_path, npz):
    d = np.load(npz, allow_pickle=True)
    con = sqlite3.connect(db_path)
    have = {n for n, in con.execute('select name from images')}
    if set(d['names'].tolist()) <= have:
        con.close()
        return
    for cid, row, params in zip(d['cam_ids'].tolist(), d['cam_rows'], d['cam_params']):
        con.execute('insert or replace into cameras(camera_id, model, width, height, params,'
                    ' prior_focal_length) values (?,?,?,?,?,?)',
                    (int(cid), int(row[0]), int(row[1]), int(row[2]), params.tobytes(),
                     int(row[3])))
    for gid, row in zip(d['rig_ids'].tolist(), d['rig_rows']):
        con.execute('insert or replace into rigs(rig_id, ref_sensor_id, ref_sensor_type)'
                    ' values (?,?,?)', (int(gid), int(row[0]), int(row[1])))
    for name, iid, cid, ks, kb, ds, dbl, fr in zip(
            d['names'].tolist(), d['image_ids'].tolist(), d['camera_ids'].tolist(),
            d['kp_shape'], d['kp_blob'], d['de_shape'], d['de_blob'], d['frame_rows']):
        con.execute('insert or replace into images(image_id, name, camera_id) values (?,?,?)',
                    (int(iid), name, int(cid)))
        con.execute('insert or replace into keypoints(image_id, rows, cols, data) values (?,?,?,?)',
                    (int(iid), int(ks[0]), int(ks[1]), kb.tobytes() or None))
        con.execute('insert or replace into descriptors(image_id, type, rows, cols, data)'
                    ' values (?,?,?,?,?)',
                    (int(iid), int(ds[0]), int(ds[1]), int(ds[2]), dbl.tobytes() or None))
        if fr[0] >= 0:
            con.execute('insert or replace into frames(frame_id, rig_id) values (?,?)',
                        (int(fr[0]), int(fr[3])))
            con.execute('insert or replace into frame_data(frame_id, data_id, sensor_id,'
                        ' sensor_type) values (?,?,?,?)',
                        (int(fr[0]), int(iid), int(fr[1]), int(fr[2])))
    con.commit()
    con.close()


def export_add_tvg(db_path, pairs, out_npz):
    """This chunk's verified two-view geometries, keyed by image NAME so parts written by
    different kernel sessions merge. Pairs are stored in lexicographic name order:
    matches[:, 0] indexes keypoints of names[i][0], matches[:, 1] of names[i][1]."""
    con = sqlite3.connect(db_path)
    ids = {n: i for i, n in con.execute('select image_id, name from images')}
    out = []
    for a, b in pairs:
        lo, hi = sorted((ids[a], ids[b]))
        r = con.execute('select rows, cols, data, config from two_view_geometries'
                        ' where pair_id=?', (lo * MAXI + hi,)).fetchone()
        if not r or not r[0] or not r[2]:
            continue
        m = np.frombuffer(r[2], np.uint32).reshape(r[0], r[1])
        n1, n2 = (a, b) if ids[a] == lo else (b, a)      # db column order follows image id
        if n1 > n2:
            n1, n2, m = n2, n1, m[:, ::-1]
        out.append((n1, n2, r[3], m))
    con.close()
    np.savez_compressed(out_npz,
                        names=np.array([(a, b, c) for a, b, c, _ in out]),
                        matches=obj_array([m for *_, m in out]),
                        attempted=np.array([f'{a} {b}' for a, b in pairs]),
                        allow_pickle=True)
    return len(out)


def add_merge_tvg(paths):
    """{(name_a, name_b): (config, matches)} over all chunk parts, plus the attempted set"""
    tvg, attempted = {}, set()
    for p in sorted(paths):
        d = np.load(p, allow_pickle=True)
        for (a, b, c), m in zip(d['names'], d['matches']):
            tvg[(a, b)] = (int(c), m)
        attempted |= {tuple(s.split(' ')) for s in d['attempted'].tolist()}
    return tvg, attempted

In [ ]:
# local preflight, before any GPU spend: (1) refit the gps->frame time offset against the
# season probe's solved poses, (2) check the geometric gate actually contains the map images
# those frames matched. The probe retrieved by NetVLAD over the 5 VIRB runs only, so recall is
# measurable on those and not on the 3 robo runs.
ADD_PROBE = Path(os.environ.get('SRS_SEASON_PROBE', '../../../../tmp/era_probe/season'))
ADD_MIN_CORR = 12


def add_gps_enu(run, head_win=10):
    """gps track in course ENU; heading over a +-1 s baseline, since differentiating 10 Hz
    virb positions frame to frame is mostly wander on the hills"""
    d = json.load(open(ADD_PATH / f'{run}.json'))
    ts = np.array([s['timestamp'] for s in d['gps_data']], float) / 1e6      # frame-clock ms
    enu = np.array(gt.ellipsoid_to_enu(
        [[s['lat'], s['long'], s['alt']] for s in d['gps_data']], LAT0, LON0, ALT0))[:, :2]
    i = np.arange(len(enu))
    a, b = np.maximum(i - head_win, 0), np.minimum(i + head_win, len(enu) - 1)
    step = enu[b] - enu[a]
    ok = np.hypot(*step.T) > 0.5
    last = np.maximum.accumulate(np.where(ok, i, -1))
    last[last < 0] = np.flatnonzero(ok)[0] if ok.any() else 0
    step = step[last]
    return d, ts, enu, np.unwrap(np.arctan2(step[:, 1], step[:, 0]))


def add_preflight(rolls, model, map_sel, radii=(15.0, 20.0, 25.0, 30.0), heads=(45, 60, 180)):
    cent = {n: s['enu'][j] for s in map_sel.values() for j, n in enumerate(s['names'])}
    mhead = {n: s['heading'][j] for s in map_sel.values() for j, n in enumerate(s['names'])}
    mid = {model.images[i].name: i for i in model.reg_image_ids()}
    has3d = {}

    def lifted(name, cols):
        if name not in has3d:
            has3d[name] = np.array([p.has_point3D() for p in model.images[mid[name]].points2D])
        return int(has3d[name][cols].sum())

    out = {}
    for run in rolls:
        d, gts, genu, ghead = add_gps_enu(run)
        assert (ADD_PATH / f'{run}.png').exists(), f'{run}: no mask staged'
        assert d['max_num_features'] == min(6000, round(4096 / (1 - d['mask_frac']))), run

        pz = np.load(ADD_PROBE / f'out/poses_{run}.npz', allow_pickle=True)
        keep = pz['localized'] & ~pz['gross']
        # probe frames are named q<roll>/<video_ms>.jpg; put them on the new frame clock
        ft = pz['t_ms'][keep] + d['camera_start'] / 1e6
        C = pz['centre'][keep][:, :2]
        grid = np.arange(-2000, 2001, 10.0)
        err = np.array([np.median(np.hypot(*(np.stack(
            [np.interp(ft + dt, gts, genu[:, j]) for j in range(2)], 1) - C).T))
            for dt in grid])
        dt = float(grid[err.argmin()])
        e = np.hypot(*(np.stack([np.interp(ft + dt, gts, genu[:, j]) for j in range(2)], 1)
                       - C).T)

        dl = np.load(ADD_PROBE / f'out/delta_{run}.npz', allow_pickle=True)
        good = defaultdict(list)
        for (a, b, _c), m in zip(dl['tvg_names'], dl['tvg_matches']):
            if len(m) and b in mid:
                n3 = lifted(b, m[:, 1])
                if n3 >= ADD_MIN_CORR:
                    good[a].append((b, n3))
        rows = []
        for r in radii:
            for h in heads:
                tot = hit = strong = strong_hit = 0
                for q, lst in good.items():
                    t = float(q.split('/')[1].split('.')[0]) + d['camera_start'] / 1e6 + dt
                    x, y = (np.interp(t, gts, genu[:, j]) for j in range(2))
                    hq = np.interp(t, gts, ghead)
                    for mn, n3 in lst:
                        dh = abs(hq - mhead[mn]) % (2 * np.pi)
                        ok = bool(np.hypot(cent[mn][0] - x, cent[mn][1] - y) <= r
                                  and np.degrees(min(dh, 2 * np.pi - dh)) <= h)
                        tot, hit = tot + 1, hit + ok
                        if n3 >= 100:
                            strong, strong_hit = strong + 1, strong_hit + ok
                rows.append({'r': r, 'head': h, 'n': tot, 'recall': round(hit / tot, 4),
                             'recall_strong': round(strong_hit / max(strong, 1), 4),
                             'n_strong': strong})
        out[run] = {'dt_ms': dt, 'err_p50': round(float(np.median(e)), 2),
                    'err_p90': round(float(np.percentile(e, 90)), 2),
                    'err_max': round(float(e.max()), 2), 'n_ref': int(keep.sum()), 'gate': rows}
        print(f'{run}: dt {dt:+.0f} ms (from {int(keep.sum())} solved poses), gps-vs-pose '
              f'p50 {np.median(e):.2f} p90 {np.percentile(e, 90):.2f} max {e.max():.2f} m')
        for x in rows:
            print(f"    r={x['r']:4.0f} head={x['head']:3d} -> recall {x['recall']:.3f} "
                  f"of {x['n']}, strong {x['recall_strong']:.3f} of {x['n_strong']}")
    return out


add_model = pycolmap.Reconstruction(str(ADD_MAP / 'model'))
add_map_sel = map_selection(add_model)
print({r: len(s['names']) for r, s in add_map_sel.items()})
add_pre = add_preflight(ADD_ROLLS, add_model, add_map_sel)
json.dump(add_pre, open(add_out / 'preflight.json', 'w'), indent=1)

In [ ]:
if ADD_SRC and not any(ADD_PATH.glob('*.json')):
    ADD_PATH.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rclone', 'copy', ADD_SRC, str(ADD_PATH), '--transfers', '4'], check=True)
add_data = add_load(ADD_PATH)
for run in ADD_ROLLS:
    extract_frames(ADD_PATH / f'{run}.MP4', add_data[run]['camera_start'], IMAGES_PATH / run,
                   window_ns=add_data[run]['roll_window_ns'])

if ADD_MAP_SRC and not (ADD_MAP / 'model').exists():
    subprocess.run(['rclone', 'copy', f'{ADD_MAP_SRC}/model', str(ADD_MAP / 'model'),
                    '--transfers', '4'], check=True)
add_map_sel = globals().get('add_map_sel') or map_selection(ADD_MAP / 'model')
add_sel = {}
for run in ADD_ROLLS:
    add_sel[run], _ = select_frames(run, add_data[run], 'virb', ADD_DELTA_S)
    s = add_sel[run]
    s['enu'][:, 2] = add_map_z(s['enu'][:, :2], add_map_sel)
    s['heading'] = add_heading(s['names'], add_data[run])
    s['source'] = np.full(len(s['names']), 'gps', dtype='<U8')
    s['sigma_m'] = np.full(len(s['names']), np.nan)
    s['section'] = np.full(len(s['names']), '', dtype='<U16')
    add_pose_file = Path(ADD_POSES) / f'{run}.npz' if ADD_POSES else None
    if add_pose_file and add_pose_file.exists():
        print(run, 'poses:', add_apply_poses(s, add_pose_file, ADD_POSE_TRUST))
print('map runs:', {r: len(s['names']) for r, s in add_map_sel.items()})

In [ ]:
add_pair_sets = {}
for run in ADD_ROLLS:
    add_pair_sets[f'intra/{run}'] = skip_pairs(add_sel[run]['names'], ADD_SKIPS)
    for mrun, ms in add_map_sel.items():
        add_pair_sets[f'map/{run}/{mrun}'] = cross_pairs(add_sel[run], ms, ADD_K_MAP,
                                                         ADD_R, ADD_HEAD_MAX)
add_pair_sets['cross/' + '/'.join(ADD_ROLLS)] = cross_pairs(
    add_sel[ADD_ROLLS[0]], add_sel[ADD_ROLLS[1]], ADD_K_PAIR, ADD_R, ADD_HEAD_MAX)
add_pairs = sorted(set().union(*add_pair_sets.values()))

for run in ADD_ROLLS:
    n = len(add_sel[run]['names'])
    per = {m.rsplit('/', 1)[1]: round(len(v) / n, 2)
           for m, v in add_pair_sets.items() if m.startswith(f'map/{run}/')}
    print(f'{run}: {n} frames | intra {len(add_pair_sets[f"intra/{run}"])} '
          f'| map {sum(len(v) for k, v in add_pair_sets.items() if k.startswith(f"map/{run}/"))} '
          f'({sum(per.values()):.1f}/frame) {per}')
add_est_min = len(add_pairs) / ADD_RATE / 60
print(f'{len(add_pairs)} unique pairs -> {add_est_min:.0f} GPU-min at {ADD_RATE} pairs/s '
      f'(cap {ADD_MAX_GPU_MIN})')
assert add_est_min <= ADD_MAX_GPU_MIN, f'estimate {add_est_min:.0f} min over budget; stopping'

for run in ADD_ROLLS:
    s = add_sel[run]
    p = add_out / f'priors_{run}.npz'
    np.savez_compressed(p, names=np.array(s['names']),
                        ts_ns=np.array([int(n.split('/')[1].split('.')[0]) for n in s['names']]),
                        enu=s['enu'], heading=s['heading'], source=s['source'],
                        sigma_m=s['sigma_m'], section=s['section'],
                        camera_start_ns=add_data[run]['camera_start'],
                        time_offset_ms=ADD_TIME_OFFSETS.get(run, 0.0),
                        focal_init=ADD_INTRINSICS[run][0], k1_init=ADD_INTRINSICS[run][1],
                        sigma_h_m=ADD_PRIOR_SIGMA_H, sigma_z_m=ADD_PRIOR_SIGMA_Z,
                        z_source='map nearest-image height')
    add_push(p, f'priors_{run}.npz')

add_manifest = {
    'map': str(ADD_MAP), 'rolls': ADD_ROLLS,
    'config': {'delta_s': ADD_DELTA_S, 'skips': list(ADD_SKIPS), 'k_map': ADD_K_MAP,
               'k_pair': ADD_K_PAIR, 'r_m': ADD_R, 'head_max_deg': ADD_HEAD_MAX,
               'chunk': ADD_CHUNK, 'lg_fp16': LG_FP16,
               'time_offsets_ms': ADD_TIME_OFFSETS, 'intrinsics': ADD_INTRINSICS},
    'poses': {'file': ADD_POSES, 'trust': list(ADD_POSE_TRUST),
              'from_file': {r: int((add_sel[r]['source'] == 'file').sum())
                            for r in ADD_ROLLS}},
    'frames': {r: len(add_sel[r]['names']) for r in ADD_ROLLS},
    'max_num_features': {r: json.load(open(ADD_PATH / f'{r}.json'))['max_num_features']
                         for r in ADD_ROLLS},
    'mask_frac': {r: json.load(open(ADD_PATH / f'{r}.json'))['mask_frac'] for r in ADD_ROLLS},
    'map_runs': {r: len(s['names']) for r, s in add_map_sel.items()},
    'classes': {k: len(v) for k, v in sorted(add_pair_sets.items())},
    'n_pairs': len(add_pairs), 'est_gpu_min': round(add_est_min, 1),
}
(add_out / 'manifest.json').write_text(json.dumps(add_manifest, indent=1))
(add_out / 'pairs.txt').write_text('\n'.join(f'{a} {b}' for a, b in add_pairs))
add_push(add_out / 'manifest.json', 'manifest.json')
add_push(add_out / 'pairs.txt', 'pairs.txt')

In [ ]:
add_work.mkdir(parents=True, exist_ok=True)
state = json.loads((add_out / 'state.json').read_text()) \
    if add_pull('state.json', add_out / 'state.json') else {}

add_db = add_work / 'work.db'
if not add_db.exists():
    t = time.time()
    shutil.copy(ADD_MAP_DB, add_db)
    add_log(f'staged map db {add_db.stat().st_size / 2 ** 30:.1f} GiB in {time.time() - t:.0f}s')
stage_models()

for run in ADD_ROLLS:
    names = add_sel[run]['names']
    con = sqlite3.connect(add_db)
    have = {n for n, in con.execute('select name from images')}
    con.close()
    if set(names) <= have:
        continue
    f = add_out / f'features_{run}.npz'
    if state.get(run, {}).get('features') and add_pull(f'features_{run}.npz', f):
        add_log(f'{run}: restoring {len(names)} features from npz')
        restore_add_features(add_db, f)
    else:
        info = json.load(open(ADD_PATH / f'{run}.json'))
        link_masks(names, mask_src=ADD_PATH / f'{run}.png')
        focal, k1 = ADD_INTRINSICS[run]
        t = time.time()
        extract_image_features(add_db, names, max_num_features=info['max_num_features'],
                               camera_params=[focal, CAMERA_PARAMS[1], CAMERA_PARAMS[2], k1])
        add_log(f'{run}: extracted {len(names)} frames in {time.time() - t:.0f}s '
                f'(budget {info["max_num_features"]}, mask_frac {info["mask_frac"]:.3f})')
        export_add_features(add_db, names, f)
        add_push(f, f'features_{run}.npz')
    state.setdefault(run, {})['features'] = True
    add_save_state(state)

ms = state.setdefault('_match', {})
ph = hashlib.sha256('\n'.join(f'{a} {b}' for a, b in add_pairs).encode()).hexdigest()[:16]
if ms.get('pair_hash') != ph:
    ms.update(pair_hash=ph, cursor=0, gpu_min=0.0)
nchunk = -(-len(add_pairs) // ADD_CHUNK)
add_log(f'matching {len(add_pairs)} pairs in {nchunk} chunks of {ADD_CHUNK}, '
        f'resuming at pair {ms["cursor"]} ({ms["gpu_min"]:.1f} GPU-min already spent)')
for c0 in range(ms['cursor'], len(add_pairs), ADD_CHUNK):
    chunk = add_pairs[c0:c0 + ADD_CHUNK]
    (add_work / 'chunk.txt').write_text('\n'.join(f'{a} {b}' for a, b in chunk))
    t = time.time()
    match_pairs(add_db, add_work / 'chunk.txt')
    part = add_out / f'tvg_{c0 // ADD_CHUNK:04d}.npz'
    n = export_add_tvg(add_db, chunk, part)
    add_push(part, part.name)
    dt = time.time() - t
    ms['cursor'], ms['gpu_min'] = c0 + len(chunk), round(ms['gpu_min'] + dt / 60, 3)
    add_save_state(state)
    add_log(f'chunk {c0 // ADD_CHUNK + 1}/{nchunk}: {len(chunk)} pairs in {dt:.0f}s '
            f'({len(chunk) / dt:.1f}/s) -> {n} verified; {ms["gpu_min"]:.1f} GPU-min used')
    if ms['gpu_min'] > ADD_MAX_GPU_MIN:
        add_log(f'BUDGET EXCEEDED at {ms["gpu_min"]:.0f} min — stopping, resume is safe')
        break
else:
    ms['done'] = True
    add_save_state(state)
    add_log(f'ALL PAIRS MATCHED in {ms["gpu_min"]:.1f} GPU-min')

In [ ]:
# local: merge the chunk parts and score every pair class. Reads only pulled artifacts, so it
# does not depend on re-running the selection locally.
add_tvg, add_attempted = add_merge_tvg(add_out.glob('tvg_[0-9]*.npz'))
add_keys = sorted(add_tvg)
np.savez_compressed(add_out / 'tvg_all.npz',
                    names=np.array([(a, b, add_tvg[(a, b)][0]) for a, b in add_keys]),
                    matches=obj_array([add_tvg[(a, b)][1] for a, b in add_keys]),
                    attempted=np.array([f'{a} {b}' for a, b in sorted(add_attempted)]),
                    allow_pickle=True)


def add_class(a, b):
    ra, rb = a.split('/')[0], b.split('/')[0]
    if ra in ADD_ROLLS and rb in ADD_ROLLS:
        return f'intra/{ra}' if ra == rb else 'cross/' + '/'.join(ADD_ROLLS)
    q, m = (a, b) if ra in ADD_ROLLS else (b, a)
    return f'map/{q.split("/")[0]}/{m.split("/")[0]}'


planned = json.loads((add_out / 'manifest.json').read_text())['classes']
by_class = defaultdict(list)
for p in add_attempted:
    by_class[add_class(*p)].append(p)
add_stats = {}
for cls, ps in sorted(by_class.items()):
    n = [len(add_tvg[p][1]) for p in ps if p in add_tvg]
    add_stats[cls] = {'pairs': planned.get(cls, len(ps)), 'attempted': len(ps),
                      'verified': len(n), 'rate': round(len(n) / max(len(ps), 1), 3),
                      'n_p50': int(np.median(n)) if n else 0,
                      'n_p90': int(np.percentile(n, 90)) if n else 0}
    s = add_stats[cls]
    print(f'{cls:34} {s["attempted"]:6} att {s["verified"]:6} ok ({s["rate"]:.2f}) '
          f'matches p50 {s["n_p50"]:5} p90 {s["n_p90"]:5}')

add_cov = {}
for run in ADD_ROLLS:
    pr = np.load(add_out / f'priors_{run}.npz', allow_pickle=True)
    names = pr['names'].tolist()
    idx = {n: i for i, n in enumerate(names)}
    nmap, nintra, ncross, best = (np.zeros(len(names), int) for _ in range(4))
    hit = [set() for _ in names]
    for (a, b), (_cfg, m) in add_tvg.items():
        for q, o in ((a, b), (b, a)):
            i = idx.get(q)
            if i is None:
                continue
            r = o.split('/')[0]
            if r == run:
                nintra[i] += 1
            elif r in ADD_ROLLS:
                ncross[i] += 1
            else:
                nmap[i] += 1
                hit[i].add(r)
            best[i] = max(best[i], len(m))
    add_cov[run] = {'names': np.array(names), 'enu': pr['enu'], 'n_map': nmap,
                    'n_intra': nintra, 'n_cross': ncross,
                    'n_map_runs': np.array([len(h) for h in hit]), 'best_matches': best}
    np.savez_compressed(add_out / f'coverage_{run}.npz', **add_cov[run])
    print(f'{run}: per frame map pairs p50 {np.median(nmap):.0f}, distinct map runs p50 '
          f'{np.median([len(h) for h in hit]):.0f}, frames with 0 map pairs '
          f'{int((nmap == 0).sum())}/{len(names)}, with <2 map runs '
          f'{int((np.array([len(h) for h in hit]) < 2).sum())}')

json.dump({'classes': add_stats,
           'gpu_min': json.loads((add_out / 'state.json').read_text())
           .get('_match', {}).get('gpu_min'),
           'n_attempted': len(add_attempted), 'n_verified': len(add_tvg)},
          open(add_out / 'matches.json', 'w'), indent=1)

## Bundle Adjustment

Add runs to the map `ADD_MAP` names: register them against it, then refit the whole thing in
one pose-prior bundle adjustment carrying the map's own robo RTK anchors at σ/8, the weights
the map itself was solved at. Nothing here writes to the map — the database is copied and the
model is written to a new directory.

`add-ba-db` → `add-ba-register` → `add-ba-gate` → `add-ba-joint` (thin, then solve) →
`add-ba-settle` (optional, off) → `add-ba-export`.

`add-ba-gate` is a frame-to-frame consistency check on the registered frames, run before the
BA rather than three review sheets later. The mapper accepts a frame on inlier count and
reprojection error, and a frame solved to the wrong place fails neither — it is self-consistent
and only breaks the chord/dt continuity of its own run. The check reports flagged frames with
their arc positions and does not act on them; `tmp/badframe` is the diagnosis and rebuild.

As checked in this is the completed 982/1005 addition, which ran against the leaf-off map now
called `rtk_spring`. `ADD_MAP_NAME` defaults to `rtk_base`, its successor: a new addition
builds on that, and reproducing this one means setting the name back to `rtk_spring`.

`fix_existing_frames=True` during registration is a cost and stability choice, not a
preservation one: PnP against static structure keeps the incremental phase cheap and stops the
map shifting under frames as they are added. The joint pass frees every frame and every point
afterwards, so nothing about the map is held in the end.

One consequence worth stating rather than leaving silent: the joint solve starts from the
structure the mapper left, not from a pristine copy of the source map's. Local BA, track
completion and filtering merged or dropped 78,285 of the map's 654,345 points during the
982/1005 registration and moved many of the survivors. That is legitimate cleanup of the same
weak population the thinning step then discards (196,111 points), and it is not undone here —
but it does mean the joint model's structure is not the source map's structure point for point.

**Every frame of a new run gets a pose prior row, at that frame's initialisation pose, with the
same σ on all of them** — `ADD_PRIOR_SIGMA_H` / `ADD_PRIOR_SIGMA_Z` = 3.0 / 10.0 m. The
position is whatever the frame was aimed and registered from: the GPS track on the plain path,
the pose file wherever `ADD_POSES` supplied one. The row is a weak regulariser holding the run
near its initialisation, *not* a per-frame expression of confidence — a bridged frame and a
GPS frame carry identical weight, and no frame goes without a row because its pose came from
somewhere unmeasured. σ_z is 10 m because `enu[:, 2]` is copied from the nearest map image and
a z inherited from the map must not come back to it as a measurement; at 10 m it cannot. This
is also what every run already in the map carries — one row per frame, σ_h = 3.0 m for the
five VIRB runs — so the new runs are constrained no differently from their neighbours. Each
new frame is still held mainly by its ~750 observations of map points.

`use_prior_position` follows those rows: it is also what stops `IterativeGlobalRefinement`
calling `Reconstruction::Normalize()` and rescaling the whole map once global BA runs at all.

**`ba_global_max_num_iterations = 1` effectively disables global BA.** One Ceres iteration is
not a solve, so `AdjustGlobalBundle` inside each global round is a no-op in practice and what
a round actually contributes is the surrounding `CompleteAndMergeTracks` → `Retriangulate` →
`FilterPoints` → `FilterFrames`. That is fine here — the one deliberate joint pass below is
the real solve — but it means any global cadence (`ba_global_frames_freq`) is not placing
optimisations. Turning global BA genuinely on means raising `ba_global_max_num_iterations`
(colmap default 50), and probably `ba_global_max_refinements` (default 5) with it.
`tmp/addrun2/AUDIT.md` §5 has the full note.

Runnable standalone: Setup → Config → Add Runs config/helpers → here.


In [ ]:
ADD_BA_WORK = Path(os.environ.get('SRS_ADDBA_WORK') or '/tmp/addba')
ADD_BA_OUT = Path('/kaggle/working/addba') if Path('/kaggle').exists() else outputs / 'addba'
ADD_BA_DST = os.environ.get('SRS_ADDBA_DST')      # rclone dir the finished db + model land in
ADD_BA_DB = ADD_BA_WORK / 'work.db'
ADD_BA_WORK.mkdir(parents=True, exist_ok=True)
ADD_BA_OUT.mkdir(parents=True, exist_ok=True)
assert ADD_BA_DB != ADD_MAP_DB, f'work on a copy; {ADD_MAP_DB} is the shipped map'

ADD_BA_BACKEND = pycolmap.BundleAdjustmentBackend.CERES
ADD_BA_GPU = False           # mapper-side ba only; its local solves are six-image, far below where gpu pays
ADD_BA_MIN_INLIERS = 15      # = the pipeline's min_num_matches; nothing shipped is below it
ADD_BA_SNAP = 200            # frames between mapper snapshots (registration resume granularity)
ADD_BA_JOINT_SCALE = 8.0     # joint pass: robo rtk priors at sigma/8, the weights the map was solved at
ADD_BA_JOINT_ITERS = 400     # as in RTK Anchoring's converged recipe (tol 1e-5, inner iterations)
ADD_BA_TEAR = os.environ.get('SRS_TEAR_SCAN')   # torn/turnaround frames whose priors were dropped
# frame-jitter gate (add-ba-gate). It reports and never removes; see tmp/badframe for the levels
# there is no margin above the clean population, so the flagged set is knob-sensitive:
# (5.0, 0.8) flags 11 chords, (5.0, 1.05) drops 3 of them, (8.0, 0.8) drops 5. treat as a
# screen, not a classifier -- only the top 6 chords separate from the clean tail
ADD_BA_GATE_MS = 5.0         # m/s over the local median. 3,426 defect-free chords peak at 4.36
ADD_BA_GATE_M = 0.8          # m of implied position error. the same chords peak at 1.02
# both off by default: 0 extra passes and no warp file reproduce the single joint solve exactly
ADD_BA_SETTLE_PASSES = 0     # extra retriangulate + BA passes after the joint solve
ADD_BA_SETTLE_M = 0.002      # stop once a pass moves the median camera centre less than this
ADD_BA_WARP_NPZ = os.environ.get('SRS_ADDBA_WARP')   # npz of `names` + `shift`, applied first
add_ba_stats = globals().get('add_ba_stats') or {}


In [ ]:
def restore_add_tvg(db_path, npz, min_inliers=15):
    """Insert the shipped two-view geometries. Stored names are in lexicographic order and
    colmap's match columns follow image id, so swap the columns when the two disagree."""
    d = np.load(npz, allow_pickle=True)
    con = sqlite3.connect(db_path)
    ids = {n: i for i, n in con.execute('select image_id, name from images')}
    out = defaultdict(int)
    for (a, b, cfg), m in zip(d['names'], d['matches']):
        if a not in ids or b not in ids:
            out['missing_image'] += 1
            continue
        if len(m) < min_inliers:
            out['below_min_inliers'] += 1
            continue
        ia, ib = ids[a], ids[b]
        m = np.ascontiguousarray(m[:, ::-1] if ia > ib else m, np.uint32)
        pair = min(ia, ib) * MAXI + max(ia, ib)
        con.execute('insert or replace into matches(pair_id, rows, cols, data) values (?,?,?,?)',
                    (pair, m.shape[0], m.shape[1], m.tobytes()))
        con.execute('insert or replace into two_view_geometries(pair_id, rows, cols, data,'
                    ' config) values (?,?,?,?,?)',
                    (pair, m.shape[0], m.shape[1], m.tobytes(), int(cfg)))
        out['inserted'] += 1
        out[f'config_{int(cfg)}'] += 1
    con.commit()
    con.close()
    return dict(out)


def add_ba_pose_priors(db_path, rolls, sigma_h=None, sigma_z=None):
    """A weak prior row on EVERY frame of the new runs, at that frame's initialisation pose.

    The position is `enu` from priors_<roll>.npz -- the pose the frame was aimed and
    registered from, whether that came from the gps track or from ADD_POSES -- and the
    covariance is the same on every row. It is a regulariser holding the run near where it
    was initialised, not a per-frame statement of confidence, so nothing about an individual
    frame changes its weight and no frame is skipped for want of a measured sigma. sigma_z is
    10 m precisely because the z is inherited from the map and must not come back to it as a
    measurement. Every run already in the map carries a row on every frame; this keeps the
    new ones alike. Existing rows are never touched, so the call is idempotent.
    """
    sigma_h = ADD_PRIOR_SIGMA_H if sigma_h is None else sigma_h
    sigma_z = ADD_PRIOR_SIGMA_Z if sigma_z is None else sigma_z
    cov = np.diag([sigma_h ** 2, sigma_h ** 2, sigma_z ** 2])
    want = {}
    for run in rolls:
        z = np.load(add_out / f'priors_{run}.npz', allow_pickle=False)
        want.update({str(n): p for n, p in zip(z['names'], z['enu'])})
    if not want:
        return {'new': 0, 'of': 0, 'covered': 0}
    with pycolmap.Database.open(str(db_path)) as db:
        have = {p.corr_data_id.id for p in db.read_all_pose_priors()}
        n = covered = 0
        for im in db.read_all_images():
            if im.name not in want:
                continue
            covered += 1
            if im.data_id.id in have:
                continue
            prior = pycolmap.PosePrior()
            prior.corr_data_id = im.data_id
            prior.position = np.asarray(want[im.name], float)
            prior.position_covariance = cov
            prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
            db.write_pose_prior(prior)
            n += 1
        total = db.num_pose_priors()
    assert covered == len(want), f'{len(want) - covered} new frames are not in the database'
    return {'new': n, 'of': len(want), 'covered': covered, 'total': total,
            'sigma_h_m': sigma_h, 'sigma_z_m': sigma_z}


def add_ba_ids(db_path, rolls):
    con = sqlite3.connect(db_path)
    ids = {n: i for i, n in con.execute('select image_id, name from images')
           if n.split('/')[0] in rolls}
    con.close()
    return ids


def add_ba_sizes(model, new_ids):
    """What the new runs brought: structure that lives only in them against structure they
    share with the map, and how many observations each side carries."""
    new = set(new_ids.values())
    reg = [i for i in model.reg_image_ids() if i in new]
    new_pts = new_obs = shared_obs = 0
    for p in model.points3D.values():
        n = sum(1 for e in p.track.elements if e.image_id in new)
        if n == 0:
            continue
        if n == p.track.length():
            new_pts += 1
            new_obs += n
        else:
            shared_obs += n
    return {'new_frames_registered': len(reg), 'new_frames_total': len(new),
            'map_frames': model.num_reg_images() - len(reg),
            'new_points': new_pts, 'other_points': len(model.points3D) - new_pts,
            'obs_on_new_points': new_obs, 'obs_on_shared_points': shared_obs}


def add_ba_move(model, ref_path):
    """How far the joint model ended up from the shipped map, frames and points separately.
    A diagnostic on the result -- everything is free in the joint solve, so motion is expected
    and only its size is informative."""
    ref = pycolmap.Reconstruction(str(ref_path))
    f = np.linalg.norm([model.images[i].projection_center() - ref.images[i].projection_center()
                        for i in ref.reg_image_ids() if model.exists_image(i)], axis=1)
    kept = [p for p in ref.point3D_ids() if model.exists_point3D(p)]
    q = np.linalg.norm([model.points3D[p].xyz - ref.points3D[p].xyz for p in kept], axis=1)
    return {'frames': {'n': len(f), 'p50': round(float(np.median(f)), 4),
                       'p90': round(float(np.percentile(f, 90)), 4),
                       'max': round(float(f.max()), 4)},
            'points': {'n': len(kept), 'dropped': ref.num_points3D() - len(kept),
                       'p50': round(float(np.median(q)), 4),
                       'p99': round(float(np.percentile(q, 99)), 4),
                       'max': round(float(q.max()), 4)}}


def add_ba_rtk_eval(model, db_path):
    """Georeference quality in the map's own currency: horizontal residual against the
    UNSCALED robo RTK priors, weighted by their per-anchor covariance (RTK Anchoring's
    convention). z is reported but judged against the DEM elsewhere, not here."""
    with pycolmap.Database.open(str(db_path)) as db:
        pri = {p.corr_data_id.id: p for p in db.read_all_pose_priors()}
    rows = [(model.images[i].projection_center() - pri[i].position,
             float(np.sqrt(pri[i].position_covariance[0, 0])))
            for i in model.reg_image_ids()
            if i in pri and not model.images[i].name.split('/')[0].isdigit()]
    e = np.array([r[0] for r in rows])
    sig = np.array([r[1] for r in rows])
    h = np.hypot(e[:, 0], e[:, 1])
    return {'n': len(rows),
            'info_weighted_m': round(float(np.sqrt((h ** 2 / sig ** 2).sum()
                                                   / (1 / sig ** 2).sum())), 4),
            'rmse_m': round(float(np.sqrt((h ** 2).mean())), 4),
            'p50_m': round(float(np.median(h)), 4),
            'norm_p50': round(float(np.median(h / sig)), 3),
            'z_rmse_m': round(float(np.sqrt((e[:, 2] ** 2).mean())), 4)}


def add_ba_thin(model, seed=0):
    """RTK Anchoring's thinning: drop track-2 points and ~40% of track<=4. The unthinned
    problem returned NO_CONVERGENCE in 999 s where the thinned one converged in ~350 s."""
    rng = np.random.default_rng(seed)
    drop = [pid for pid, p in model.points3D.items()
            if p.track.length() <= 2 or (p.track.length() <= 4 and rng.random() > 0.6)]
    for pid in drop:
        model.delete_point3D(pid)
    return len(drop)


def add_ba_joint(model, db_path, scale=None, iters=None, tol=1e-5):
    """Joint pass, following tmp/colab_loc/converge_final.py. Everything free; the map's own
    robo RTK priors carry the georeference at sigma/scale, the weights it was solved at.
    Tightening is by name prefix, so numeric VIRB rolls keep their loose covariance; 982 and
    1005 carry no prior row at all, which is deliberate -- their z is copied from the nearest
    map image and any finite sigma_z would feed the map back to itself. Intrinsics frozen:
    freeing them adds a scale/z drift mode."""
    scale = ADD_BA_JOINT_SCALE if scale is None else scale
    drops = set()
    if ADD_BA_TEAR and Path(ADD_BA_TEAR).exists():
        drops = {n for n, v in json.load(open(ADD_BA_TEAR)).items() if v['split'] > 30}
    with pycolmap.Database.open(str(db_path)) as db:
        name_of = {im.data_id.id: im.name for im in db.read_all_images()}
        priors = []
        for pr in db.read_all_pose_priors():
            i = pr.corr_data_id.id
            n = name_of[i]
            if n in drops or not model.exists_image(i) or not model.images[i].has_pose:
                continue
            if not n.split('/')[0].isdigit():
                pr.position_covariance = pr.position_covariance / scale ** 2
            priors.append(pr)
    cfg = pycolmap.BundleAdjustmentConfig()
    for i in model.reg_image_ids():
        cfg.add_image(i)
    o = pycolmap.BundleAdjustmentOptions()
    o.refine_focal_length = False
    o.refine_extra_params = False
    o.refine_principal_point = False
    o.refine_sensor_from_rig = False
    so = o.ceres.solver_options
    so.max_num_iterations = ADD_BA_JOINT_ITERS if iters is None else iters
    so.num_threads = -1
    so.function_tolerance = tol
    so.use_inner_iterations = True
    t = time.time()
    summary = pycolmap.create_pose_prior_ceres_bundle_adjuster(
        o, pycolmap.PosePriorBundleAdjustmentOptions(), cfg, priors, model).solve()
    return summary, time.time() - t, {'priors': len(priors), 'dropped_torn': len(drops)}


def add_ba_gate(model, thresh=None, excess=None, half=1.0):
    """Frame-to-frame consistency of every run in the model: chord / dt against its own
    +-half s running median. A frame solved to the wrong place passes colmap's inlier and
    reprojection tests -- the mapper checks neither trajectory continuity nor timestamps --
    but breaks this.

    A chord is flagged only when it is BOTH `thresh` m/s over that median and `excess` m of
    implied position inconsistency. Frame spacing here is irregular (p50 ~0.2-0.34 s, p5 one
    or two video frames), so the m/s form alone fires on 16-70 ms gaps where the real
    disagreement is a few centimetres; those are counted as `n_speed_only`.

    Two consecutive flagged chords are ONE displaced frame, not two: same sign is a lateral
    excursion, opposite signs a displacement along the track.
    """
    thresh = ADD_BA_GATE_MS if thresh is None else thresh
    excess = ADD_BA_GATE_M if excess is None else excess
    by = {}
    for i in model.reg_image_ids():
        n = model.images[i].name
        try:
            ns = int(n.rsplit('/', 1)[-1].rsplit('.', 1)[0])
        except ValueError:
            continue
        by.setdefault(n.split('/')[0], []).append((ns, i))
    rep = {'thresh_m_s': thresh, 'excess_m': excess, 'half_s': half,
           'n_images': int(model.num_reg_images()), 'runs': {}, 'flagged': []}
    for run, v in sorted(by.items()):
        if len(v) < 5:
            continue
        v.sort()
        ids = [i for _, i in v]
        t = np.array([q for q, _ in v], float) / 1e9
        C = np.array([model.images[i].projection_center() for i in ids])
        step = np.linalg.norm(np.diff(C, axis=0), axis=1)
        dt = np.diff(t)
        vs = step / dt
        tm = 0.5 * (t[1:] + t[:-1])
        r = vs - np.array([np.median(vs[np.abs(tm - u) <= half]) for u in tm])
        ex = r * dt
        arc = np.concatenate([[0.0], np.cumsum(step)])
        hot = np.abs(r) > thresh
        bad = list(np.flatnonzero(hot & (np.abs(ex) > excess)))
        rep['runs'][run] = {'n': len(t), 'rms': round(float(np.sqrt(np.mean(r ** 2))), 3),
                            'p95': round(float(np.percentile(np.abs(r), 95)), 3),
                            'n_over_thresh': int(hot.sum()), 'n_flagged': len(bad),
                            'n_speed_only': int(hot.sum()) - len(bad)}
        k = 0
        while k < len(bad):
            j = int(bad[k])
            pair = k + 1 < len(bad) and bad[k + 1] == j + 1
            rep['flagged'].append({
                'run': run,
                'frame': model.images[ids[j + 1]].name if pair else None,
                'arc_m': round(float(arc[j + 1] if pair else arc[j]), 1),
                'signature': None if not pair else
                             'lateral' if r[j] * r[j + 1] > 0 else 'along-track',
                'chords': [{'from': model.images[ids[q]].name,
                            'to': model.images[ids[q + 1]].name,
                            'arc_m': round(float(arc[q]), 1),
                            'dt_s': round(float(dt[q]), 4),
                            'len_m': round(float(step[q]), 3),
                            'r_m_s': round(float(r[q]), 2),
                            'excess_m': round(float(ex[q]), 3)}
                           for q in ([j, j + 1] if pair else [j])]})
            k += 2 if pair else 1
    rep['n_flagged'] = len(rep['flagged'])
    return rep


def add_ba_gate_print(rep):
    print(f'frame-jitter gate: {rep["n_images"]} images, {rep["n_flagged"]} flagged '
          f'(>{rep["thresh_m_s"]} m/s and >{rep["excess_m"]} m)')
    for run, v in rep['runs'].items():
        print(f'  {run:26s} n={v["n"]:4d} rms {v["rms"]:6.3f} p95 {v["p95"]:5.3f} '
              f'over-thresh {v["n_over_thresh"]:3d} flagged {v["n_flagged"]:2d} '
              f'speed-only {v["n_speed_only"]:2d}')
    for f in rep['flagged']:
        c = f['chords'][0]
        print(f'  FLAG {f["run"]} arc {f["arc_m"]:.1f} m  '
              f'{f["frame"] or c["from"] + " | " + c["to"]}'
              + (f' ({f["signature"]})' if f['signature'] else '') + '  '
              + ' '.join(f'[dt {q["dt_s"]:.3f}s len {q["len_m"]:.2f}m r {q["r_m_s"]:+.1f} '
                         f'excess {q["excess_m"]:+.2f}m]' for q in f['chords']))


def add_ba_shift_frames(model, npz_path):
    """Translate camera centres by a precomputed per-image shift, keeping their rotations.

    The along-track correction field is built from the racebox doppler against the course arc
    axis, neither of which exists in this environment, so it is computed in the analysis tree
    (tmp/badframe/b11_warp.py) and only the resulting `names` + `shift` table is carried here.
    """
    z = np.load(npz_path, allow_pickle=True)
    idx = {model.images[i].name: i for i in model.reg_image_ids()}
    moved = []
    for name, d in zip(z['names'].tolist(), np.asarray(z['shift'], float)):
        i = idx.get(name)
        if i is None:
            continue
        fr = model.frames[model.images[i].frame_id]
        rfw = fr.rig_from_world
        R = rfw.rotation.matrix()
        r2 = type(rfw)()
        r2.rotation = rfw.rotation
        r2.translation = -(R @ (-(R.T @ rfw.translation) + d))
        fr.rig_from_world = r2
        moved.append(float(np.linalg.norm(d)))
    return {'n': len(moved), 'rms_m': round(float(np.sqrt(np.mean(np.square(moved)))), 4),
            'max_m': round(float(np.max(moved)), 4), 'of': len(z['names'])}


def add_ba_settle(model, db_path, work, passes=None, settle_m=None):
    """Repeat retriangulate -> thin -> pose-prior BA until the model stops moving.

    Off by default. The frames an addition brings in are registered against a frozen map and
    given one joint solve, so their structure is newer than the rest and a further pass still
    moves it; tmp/warpfix measured 3.1 cm of median centre motion from a second pass with
    nothing else changed. Whether that is worth the ~45 min a pass costs is a judgement the
    numbers in tmp/badframe are there to inform.
    """
    passes = ADD_BA_SETTLE_PASSES if passes is None else passes
    settle_m = ADD_BA_SETTLE_M if settle_m is None else settle_m
    empty = Path('/tmp/noimgs')
    empty.mkdir(exist_ok=True)
    out = []
    for k in range(passes):
        before = {i: model.images[i].projection_center().copy() for i in model.reg_image_ids()}
        tri = work / f'tri_settle{k}'
        shutil.rmtree(tri, ignore_errors=True)
        tri.mkdir(parents=True)
        t = time.time()
        model = pycolmap.triangulate_points(model, str(db_path), str(empty), str(tri),
                                            clear_points=True)
        tri_s = time.time() - t
        thinned = add_ba_thin(model)
        summary, secs, info = add_ba_joint(model, db_path)
        mv = np.linalg.norm([model.images[i].projection_center() - c
                             for i, c in before.items() if model.exists_image(i)
                             and model.images[i].has_pose], axis=1)
        row = {'pass': k + 1, 'tri_s': round(tri_s, 1), 'ba_s': round(secs, 1),
               'thinned': thinned, 'points': model.num_points3D(),
               'moved_p50': round(float(np.median(mv)), 5),
               'moved_p90': round(float(np.percentile(mv, 90)), 5),
               'termination': str(summary.termination_type)}
        out.append(row)
        shutil.rmtree(tri, ignore_errors=True)
        print(f'settle pass {k + 1}: moved p50 {row["moved_p50"]:.5f} m '
              f'p90 {row["moved_p90"]:.5f} m, {row["points"]:,} points')
        if row['moved_p50'] < settle_m:
            row['settled'] = True
            break
    return model, out


def add_ba_report(model, new_ids, tag=''):
    """Per-run registration, reprojection error and gps-prior residual (a check, not a term)."""
    rows = {}
    for run in ADD_ROLLS:
        pr = np.load(add_out / f'priors_{run}.npz', allow_pickle=True)
        pri = dict(zip(pr['names'].tolist(), pr['enu']))
        ids = [i for n, i in new_ids.items() if n.split('/')[0] == run]
        reg = [i for i in ids if model.exists_image(i) and model.images[i].has_pose]
        e = np.array([model.images[i].projection_center() - pri[model.images[i].name]
                      for i in reg])
        h = np.hypot(e[:, 0], e[:, 1]) if len(e) else np.zeros(0)
        cam = model.cameras[model.images[reg[0]].camera_id] if reg else None
        rows[run] = {'registered': len(reg), 'of': len(ids),
                     'gps_p50': round(float(np.median(h)), 2) if len(h) else None,
                     'gps_p90': round(float(np.percentile(h, 90)), 2) if len(h) else None,
                     'focal': round(float(cam.params[0]), 2) if cam else None,
                     'k1': round(float(cam.params[3]), 4) if cam else None}
        print(f'{tag}{run}: {len(reg)}/{len(ids)} registered | vs gps prior p50 '
              f'{rows[run]["gps_p50"]} p90 {rows[run]["gps_p90"]} m | focal '
              f'{rows[run]["focal"]} k1 {rows[run]["k1"]}')
    return rows


In [ ]:
# the work db is a copy: ADD_MAP_DB is the shipped map's own db and is only ever read, here
if not ADD_BA_DB.exists():
    t = time.time()
    shutil.copy(ADD_MAP_DB, ADD_BA_DB)
    print(f'staged map db {ADD_BA_DB.stat().st_size / 2 ** 30:.1f} GiB in {time.time() - t:.0f}s')

for run in ADD_ROLLS:
    restore_add_features(ADD_BA_DB, add_out / f'features_{run}.npz')
add_ba_stats['tvg'] = restore_add_tvg(ADD_BA_DB, add_out / 'tvg_all.npz', ADD_BA_MIN_INLIERS)
add_ba_stats['priors'] = add_ba_pose_priors(ADD_BA_DB, ADD_ROLLS)
add_ba_new = add_ba_ids(ADD_BA_DB, ADD_ROLLS)

con = sqlite3.connect(ADD_BA_DB)
add_ba_stats['db'] = {t: con.execute(f'select count(*) from {t}').fetchone()[0]
                      for t in ('images', 'cameras', 'rigs', 'frames', 'keypoints',
                                'descriptors', 'two_view_geometries', 'pose_priors')}
add_ba_stats['db']['keypoints_new'] = con.execute(
    'select sum(rows) from keypoints where image_id >= ?', (min(add_ba_new.values()),)).fetchone()[0]
con.close()
print(add_ba_stats['db'])
print(add_ba_stats['tvg'])
print(add_ba_stats['priors'])
print(f'{len(add_ba_new)} new images, ids {min(add_ba_new.values())}-{max(add_ba_new.values())}')


In [ ]:
# register the new frames against the map. fix_existing_frames pins the map's frames for the
# incremental phase only: PnP against static structure is cheap and keeps the map from shifting
# under frames as they are added. It is not a preservation measure -- the joint pass frees
# everything afterwards, and the mapper's merging and filtering of map points stands.
# The mapper's periodic global BA is pushed out of reach for the same reasons: each trigger is
# a ~14 M-residual whole-map solve (59% of mapper time in the logged robo run) and the one
# deliberate joint solve in the next cell replaces it.
add_ba_opt = {
    'fix_existing_frames': True,
    'multiple_models': False,
    'ba_refine_principal_point': False,
    'ba_refine_sensor_from_rig': False,
    'ba_use_gpu': ADD_BA_GPU,
    'ba_local_backend': ADD_BA_BACKEND,
    'ba_global_backend': ADD_BA_BACKEND,
    'ba_global_frames_ratio': 10.0,
    'ba_global_points_ratio': 10.0,
    'ba_global_frames_freq': 10 ** 6,
    'ba_global_points_freq': 10 ** 9,
    # 1 iteration makes the AdjustGlobalBundle inside a global round a no-op; only the
    # retriangulate/filter around it acts. Raise it (colmap default 50) to enable global BA.
    'ba_global_max_num_iterations': 1,
    'ba_global_max_refinements': 1,
    # on when the new frames carry prior rows: those poses are their initialisation, and the
    # prior adjuster replaces the global refinement's map-rescaling Normalize()
    'use_prior_position': bool(add_ba_stats.get('priors', {}).get('covered')),
    'snapshot_path': str(ADD_BA_WORK / 'snap'),
    'snapshot_frames_freq': ADD_BA_SNAP,
    'mapper': {'max_reg_trials': 2},
}
(ADD_BA_WORK / 'snap').mkdir(parents=True, exist_ok=True)
add_ba_reg = ADD_BA_WORK / 'registered'
# resume from the most recent model this pipeline wrote, else the map itself
cand = [p for p in [*(ADD_BA_WORK / 'snap').glob('*'), add_ba_reg] if (p / 'images.bin').exists()]
add_ba_in = max(cand, key=lambda p: (p / 'images.bin').stat().st_mtime) if cand else ADD_MAP / 'model'
print('input model:', add_ba_in)

t = time.time()
with pycolmap.ostream():
    add_ba_recs = pycolmap.incremental_mapping(ADD_BA_DB, IMAGES_PATH, add_ba_reg,
                                               options=add_ba_opt, input_path=str(add_ba_in))
add_ba_model = add_ba_recs[0]
add_ba_stats['register_s'] = round(time.time() - t, 1)
add_ba_new = add_ba_ids(ADD_BA_DB, ADD_ROLLS)
add_ba_stats['registered'] = add_ba_report(add_ba_model, add_ba_new)
add_ba_stats['sizes_after_register'] = add_ba_sizes(add_ba_model, add_ba_new)
print(f'registration {add_ba_stats["register_s"] / 60:.1f} min',
      add_ba_stats['sizes_after_register'])


In [ ]:
# frame-to-frame consistency of the registered frames, before the BA spends an hour on them.
# colmap's mapper checks inliers and reprojection, neither of which sees a frame solved to the
# wrong place: it stays self-consistent and only breaks the continuity of its own run. Report
# only -- tmp/badframe diagnoses a flag and rebuilds the model without the frames it blames.
add_ba_stats['gate'] = add_ba_gate(add_ba_model)
add_ba_gate_print(add_ba_stats['gate'])


In [ ]:
# one deliberate joint pass over the COMPLETE registered set -- never on the mapper's frequency
# counter, which would refit part-way and leave no record of which frames saw which structure.
# It starts from the structure registration left, mapper merging and filtering included, and
# mutates add_ba_model in place: reload ADD_BA_WORK / 'registered' to run it again.
add_ba_jstats = {**add_ba_stats,
                 'rtk_before': add_ba_rtk_eval(add_ba_model, ADD_BA_DB),
                 'new_runs_before': add_ba_report(add_ba_model, add_ba_new, 'registered ')}
add_ba_jstats['thinned'] = add_ba_thin(add_ba_model)
add_ba_jsum, add_ba_js, add_ba_jn = add_ba_joint(add_ba_model, ADD_BA_DB)
add_ba_jstats['joint'] = {'seconds': round(add_ba_js, 1), **add_ba_jn,
                          'residuals': add_ba_jsum.num_residuals,
                          'termination': str(add_ba_jsum.termination_type),
                          'report': add_ba_jsum.brief_report()}
add_ba_jstats['rtk_after'] = add_ba_rtk_eval(add_ba_model, ADD_BA_DB)
add_ba_jstats['map_moved'] = add_ba_move(add_ba_model, ADD_MAP / 'model')
add_ba_jstats['new_runs_after'] = add_ba_report(add_ba_model, add_ba_new, 'joint ')
print(f'joint {add_ba_js / 60:.1f} min', json.dumps(add_ba_jstats['joint'], indent=1))
print('rtk before', add_ba_jstats['rtk_before'])
print('rtk after ', add_ba_jstats['rtk_after'])
print('map moved ', add_ba_jstats['map_moved'])


In [ ]:
# optional, off by default: ADD_BA_SETTLE_PASSES = 0 and no ADD_BA_WARP_NPZ reproduce the
# single joint solve above exactly. Both exist because tmp/badframe measured them; the numbers
# for this map are in tmp/badframe/findings.json, and turning either on is a deliberate choice.
if ADD_BA_WARP_NPZ:
    add_ba_jstats['warp'] = add_ba_shift_frames(add_ba_model, ADD_BA_WARP_NPZ)
    print('warp applied:', add_ba_jstats['warp'])
if ADD_BA_WARP_NPZ or ADD_BA_SETTLE_PASSES:
    add_ba_model, add_ba_jstats['settle'] = add_ba_settle(
        add_ba_model, ADD_BA_DB, ADD_BA_WORK,
        passes=max(ADD_BA_SETTLE_PASSES, 1 if ADD_BA_WARP_NPZ else 0))
    add_ba_jstats['rtk_settled'] = add_ba_rtk_eval(add_ba_model, ADD_BA_DB)
    add_ba_stats['gate_settled'] = add_ba_gate(add_ba_model)
    add_ba_gate_print(add_ba_stats['gate_settled'])
    print('rtk after settling', add_ba_jstats['rtk_settled'])


In [ ]:
add_ba_dir = ADD_BA_OUT / 'model_joint'
add_ba_dir.mkdir(parents=True, exist_ok=True)
add_ba_model.write(add_ba_dir)
(ADD_BA_OUT / 'stats_joint.json').write_text(json.dumps(add_ba_jstats, indent=1))
print('joint ->', add_ba_dir)
if ADD_BA_DST:
    subprocess.run(['rclone', 'copy', str(add_ba_dir), f'{ADD_BA_DST}/model_joint',
                    '--transfers', '4'], check=True)
    subprocess.run(['rclone', 'copyto', str(ADD_BA_OUT / 'stats_joint.json'),
                    f'{ADD_BA_DST}/stats_joint.json'], check=True)
# the work db is reproducible from ADD_MAP_DB + the addmatch npz in minutes;
# push it only when asked for
if ADD_BA_DST and os.environ.get('SRS_ADDBA_PUSH_DB'):
    subprocess.run(['rclone', 'copyto', str(ADD_BA_DB), f'{ADD_BA_DST}/database.db'], check=True)
